In [0]:

from pyspark.sql import functions as F

print("Project: RetailRecon")
print("Spark version:", spark.version)

check_df = spark.createDataFrame(
    [
        ("environment_check", "SUCCESS"),
        ("pyspark_available", "SUCCESS"),
    ],
    ["check_name", "status"]
)

display(check_df)

In [0]:

from pathlib import Path

file_path = "/Volumes/portfolio_projects/retail_sales/raw/online_retail_II.xlsx"
workbook_path = Path(file_path)

# 1. Verify the source file
if not workbook_path.is_file():
    raise FileNotFoundError(
        f"Workbook not found at: {file_path}\n"
        "Check the catalog, schema, volume, and filename."
    )

file_size_mb = workbook_path.stat().st_size / (1024 * 1024)

print("Source file verified ✅")
print(f"Path: {file_path}")
print(f"File size: {file_size_mb:.2f} MB")

# 2. Inspect workbook metadata only
try:
    from openpyxl import load_workbook

    workbook = load_workbook(
        filename=file_path,
        read_only=True,
        data_only=True
    )

    print("\nWorkbook worksheets:")
    for worksheet in workbook.worksheets:
        print(
            f"Sheet: {worksheet.title!r} | "
            f"Rows (including header): {worksheet.max_row:,} | "
            f"Columns: {worksheet.max_column:,}"
        )

    workbook.close()

except ImportError:
    print(
        "\nopenpyxl is not available in this environment. "
        "Do not load the full workbook with pandas yet."
    )

In [0]:

from pathlib import Path
from hashlib import sha256
from datetime import date, datetime
from openpyxl import load_workbook
import csv

# --------------------------------------------------
# 1. Source and output paths
# --------------------------------------------------
SOURCE_PATH = (
    "/Volumes/portfolio_projects/retail_sales/raw/"
    "online_retail_II.xlsx"
)

STAGING_DIR = (
    "/Volumes/portfolio_projects/retail_sales/raw/"
    "_retail_recon_staging"
)

source_file = Path(SOURCE_PATH)
staging_dir = Path(STAGING_DIR)

if not source_file.is_file():
    raise FileNotFoundError(f"Source workbook not found: {SOURCE_PATH}")

staging_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. Calculate a stable source-file checksum
# --------------------------------------------------
hasher = sha256()

with source_file.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        hasher.update(chunk)

workbook_sha256 = hasher.hexdigest()

print("Workbook SHA-256:", workbook_sha256)

# --------------------------------------------------
# 3. Convert both sheets row by row
# --------------------------------------------------
expected_sheets = [
    "Year 2009-2010",
    "Year 2010-2011",
]

workbook = load_workbook(
    filename=SOURCE_PATH,
    read_only=True,
    data_only=True,
)

try:
    missing_sheets = set(expected_sheets) - set(workbook.sheetnames)
    if missing_sheets:
        raise ValueError(f"Missing worksheets: {sorted(missing_sheets)}")

    for sheet_name in expected_sheets:
        worksheet = workbook[sheet_name]

        # Read the original column headers.
        rows = worksheet.iter_rows(values_only=True)
        original_headers = next(rows, None)

        if not original_headers:
            raise ValueError(f"No header row found in {sheet_name!r}")

        headers = [
            str(value).strip() if value is not None
            else f"_unnamed_{index}"
            for index, value in enumerate(original_headers)
        ]

        output_path = staging_dir / (
            sheet_name.replace(" ", "_").replace("-", "_") + ".csv"
        )

        rows_written = 0

        with output_path.open(
            "w",
            encoding="utf-8",
            newline="",
        ) as output_file:
            writer = csv.writer(output_file)

            writer.writerow(
                headers
                + [
                    "source_sheet",
                    "source_row_number",
                    "source_workbook_sha256",
                ]
            )

            # Excel row 1 is the header, so data starts at row 2.
            for excel_row_number, values in enumerate(rows, start=2):

                converted_values = []

                for value in values:
                    if isinstance(value, datetime):
                        converted_values.append(
                            value.isoformat(sep=" ")
                        )
                    elif isinstance(value, date):
                        converted_values.append(value.isoformat())
                    elif value is None:
                        converted_values.append("")
                    else:
                        converted_values.append(str(value))

                writer.writerow(
                    converted_values
                    + [
                        sheet_name,
                        excel_row_number,
                        workbook_sha256,
                    ]
                )

                rows_written += 1

        print(
            f"Completed: {sheet_name} | "
            f"Rows written: {rows_written:,} | "
            f"File: {output_path}"
        )

finally:
    workbook.close()

print("\nSource extraction completed.")
print("No overlap filtering or business transformations performed.")

In [0]:

from pathlib import Path
import csv

staging_dir = Path(
    "/Volumes/portfolio_projects/retail_sales/raw/"
    "_retail_recon_staging"
)

expected_counts = {
    "Year_2009_2010.csv": 525_461,
    "Year_2010_2011.csv": 541_910,
}

for filename, expected_count in expected_counts.items():
    path = staging_dir / filename

    if not path.is_file():
        raise FileNotFoundError(f"Missing output: {path}")

    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        headers = next(reader)

        actual_count = sum(1 for _ in reader)

    print(f"\nFile: {filename}")
    print(f"Data rows: {actual_count:,}")
    print(f"Expected:  {expected_count:,}")
    print(f"Columns:   {len(headers)}")

    assert actual_count == expected_count, (
        f"Row-count mismatch in {filename}"
    )

    assert "source_sheet" in headers
    assert "source_row_number" in headers
    assert "source_workbook_sha256" in headers

print("\nPASS: both source extracts passed validation.")

In [0]:

from pyspark.sql import functions as F
import re

# ============================================================
# RetailRecon | Raw Source -> Parquet
# Environment: Databricks development
# Production deployment: NOT used
# ============================================================

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SOURCE_DIR = (
    "/Volumes/portfolio_projects/retail_sales/raw/"
    "_retail_recon_staging"
)

PARQUET_PATH = (
    "/Volumes/portfolio_projects/retail_sales/raw/"
    "source_raw_parquet"
)

SOURCE_FILES = [
    {
        "path": f"{SOURCE_DIR}/Year_2009_2010.csv",
        "sheet": "Year 2009-2010",
        "expected_rows": 525_461,
    },
    {
        "path": f"{SOURCE_DIR}/Year_2010_2011.csv",
        "sheet": "Year 2010-2011",
        "expected_rows": 541_910,
    },
]

EXPECTED_TOTAL_ROWS = sum(
    source["expected_rows"] for source in SOURCE_FILES
)

REQUIRED_COLUMNS = {
    "invoice",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "price",
    "customer_id",
    "country",
    "source_sheet",
    "source_row_number",
    "source_workbook_sha256",
}

print("=" * 65)
print("RetailRecon: Raw Source -> Parquet")
print("=" * 65)


# ------------------------------------------------------------
# 2. Normalize source column names
# ------------------------------------------------------------

def normalize_column_name(name: str) -> str:
    """
    Normalize Excel headers to lowercase snake_case.

    Examples:
        StockCode  -> stock_code
        InvoiceDate -> invoice_date
        Customer ID -> customer_id
        Invoice     -> invoice
    """
    name = name.strip()

    # Split PascalCase and camelCase.
    name = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        name,
    )

    # Replace spaces and punctuation with underscores.
    name = re.sub(
        r"[^a-zA-Z0-9]+",
        "_",
        name,
    )

    # Collapse repeated underscores and lowercase.
    name = re.sub(
        r"_+",
        "_",
        name,
    ).strip("_")

    return name.lower()


# Verify the normalization logic before processing the files.
normalization_tests = {
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "price",
    "Customer ID": "customer_id",
    "Country": "country",
}

for source_name, expected_name in normalization_tests.items():
    actual_name = normalize_column_name(source_name)

    if actual_name != expected_name:
        raise ValueError(
            f"Column normalization failed: {source_name} "
            f"became {actual_name}, expected {expected_name}"
        )

print("\nPASS: Column normalization checks")


# ------------------------------------------------------------
# 3. Read each CSV as strings and validate its schema
# ------------------------------------------------------------

source_frames = []

for source in SOURCE_FILES:
    csv_path = source["path"]
    expected_sheet = source["sheet"]
    expected_rows = source["expected_rows"]

    print(f"\nReading: {csv_path}")

    # Keep original values as strings in the raw layer.
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "FAILFAST")
        .option("encoding", "UTF-8")
        .option("quote", '"')
        .option("escape", '"')
        .load(csv_path)
    )

    # Rename columns safely.
    normalized_names = [
        normalize_column_name(column)
        for column in df.columns
    ]

    # Prevent different source headers from collapsing
    # into the same normalized name.
    if len(normalized_names) != len(set(normalized_names)):
        raise ValueError(
            f"Duplicate column names after normalization: {csv_path}\n"
            f"Normalized columns: {normalized_names}"
        )

    for old_name, new_name in zip(df.columns, normalized_names):
        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)

    # Check the schema before doing any transformations.
    missing_columns = REQUIRED_COLUMNS - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing expected columns in {csv_path}: "
            f"{sorted(missing_columns)}\n"
            f"Actual columns: {df.columns}"
        )

    print("PASS: Required columns found")

    # Confirm that the source file has the expected worksheet label.
    observed_sheets = {
        row["source_sheet"]
        for row in (
            df.select("source_sheet")
            .distinct()
            .collect()
        )
    }

    if observed_sheets != {expected_sheet}:
        raise ValueError(
            f"Unexpected source_sheet values in {csv_path}. "
            f"Expected: {expected_sheet!r}; "
            f"found: {observed_sheets}"
        )

    print(f"PASS: Source sheet = {expected_sheet}")

    # Convert the Excel row number to a numeric type.
    df = df.withColumn(
        "source_row_number",
        F.col("source_row_number").cast("long"),
    )

    # Validate required lineage before generating row IDs.
    invalid_lineage = df.filter(
        F.col("source_sheet").isNull()
        | F.col("source_row_number").isNull()
        | F.col("source_workbook_sha256").isNull()
        | (F.trim(F.col("source_workbook_sha256")) == "")
    )

    if invalid_lineage.limit(1).count() > 0:
        raise ValueError(
            f"Missing or invalid source lineage in {csv_path}"
        )

    # A source row is identified by its workbook checksum,
    # worksheet and original Excel row number.
    df = df.withColumn(
        "row_id",
        F.sha2(
            F.to_json(
                F.struct(
                    F.col("source_workbook_sha256").alias("workbook"),
                    F.col("source_sheet").alias("sheet"),
                    F.col("source_row_number").alias("excel_row"),
                )
            ),
            256,
        ),
    )

    # Verify the original source-row count.
    actual_rows = df.count()

    print(f"Expected source rows: {expected_rows:,}")
    print(f"Actual source rows:   {actual_rows:,}")

    if actual_rows != expected_rows:
        raise ValueError(
            f"Row-count mismatch for {expected_sheet}: "
            f"expected {expected_rows:,}; found {actual_rows:,}"
        )

    print("PASS: Source row count")

    source_frames.append(df)


# ------------------------------------------------------------
# 4. Combine both worksheets without filtering any rows
# ------------------------------------------------------------

raw_source = source_frames[0].unionByName(source_frames[1])

actual_source_sheets = {
    row["source_sheet"]
    for row in (
        raw_source.select("source_sheet")
        .distinct()
        .collect()
    )
}

expected_source_sheets = {
    source["sheet"] for source in SOURCE_FILES
}

if actual_source_sheets != expected_source_sheets:
    raise ValueError(
        f"Unexpected source sheets: {actual_source_sheets}"
    )

print("\nBoth source worksheets are combined.")


# ------------------------------------------------------------
# 5. Validate lineage before writing
# ------------------------------------------------------------

lineage_errors = raw_source.filter(
    F.col("row_id").isNull()
    | F.col("source_sheet").isNull()
    | F.col("source_row_number").isNull()
    | F.col("source_workbook_sha256").isNull()
)

if lineage_errors.limit(1).count() > 0:
    raise ValueError(
        "Lineage validation failed. Parquet write cancelled."
    )

print("PASS: Required lineage values are present")


# ------------------------------------------------------------
# 6. Write the raw Parquet dataset
# ------------------------------------------------------------

print(f"\nWriting Parquet to:\n{PARQUET_PATH}")

(
    raw_source.write
    .format("parquet")
    .mode("overwrite")
    .option("compression", "snappy")
    .save(PARQUET_PATH)
)

print("\n" + "=" * 65)
print("SUCCESS: Raw Parquet dataset written")
print(f"Output path: {PARQUET_PATH}")
print(f"Expected total rows: {EXPECTED_TOTAL_ROWS:,}")
print("=" * 65)

In [0]:

from pyspark.sql import functions as F

# ============================================================
# RetailRecon | Post-write Parquet Validation
# ============================================================

PARQUET_PATH = (
    "/Volumes/portfolio_projects/retail_sales/raw/"
    "source_raw_parquet"
)

EXPECTED_ROWS = {
    "Year 2009-2010": 525_461,
    "Year 2010-2011": 541_910,
}

EXPECTED_TOTAL = sum(EXPECTED_ROWS.values())

saved_df = spark.read.parquet(PARQUET_PATH)

# 1. Total rows
total_rows = saved_df.count()

print("Total persisted rows:", f"{total_rows:,}")

assert total_rows == EXPECTED_TOTAL, (
    f"Total row-count mismatch: "
    f"{total_rows:,} != {EXPECTED_TOTAL:,}"
)

# 2. Per-worksheet counts
sheet_counts = {
    row["source_sheet"]: row["count"]
    for row in (
        saved_df.groupBy("source_sheet")
        .count()
        .collect()
    )
}

for sheet, expected in EXPECTED_ROWS.items():
    actual = sheet_counts.get(sheet, 0)

    print(
        f"{sheet}: actual={actual:,}, "
        f"expected={expected:,}"
    )

    assert actual == expected, (
        f"Row-count mismatch for {sheet}"
    )

# 3. Validate mandatory lineage
bad_lineage_count = saved_df.filter(
    F.col("row_id").isNull()
    | F.col("source_sheet").isNull()
    | F.col("source_row_number").isNull()
    | F.col("source_workbook_sha256").isNull()
).count()

assert bad_lineage_count == 0, (
    f"Rows with missing lineage: {bad_lineage_count:,}"
)

# 4. Validate row_id uniqueness
unique_row_ids = saved_df.select("row_id").distinct().count()

print("Unique row IDs:", f"{unique_row_ids:,}")

assert unique_row_ids == total_rows, (
    f"Duplicate row IDs detected: "
    f"{total_rows - unique_row_ids:,}"
)

# 5. Validate lineage for each sheet
invalid_row_numbers = saved_df.filter(
    (F.col("source_row_number") < 2)
    | (F.col("source_row_number").isNull())
).count()

assert invalid_row_numbers == 0, (
    f"Invalid Excel row numbers: {invalid_row_numbers:,}"
)

print("\n" + "=" * 55)
print("PASS: RAW PARQUET VALIDATION")
print("=" * 55)
print("Total rows:          ", f"{total_rows:,}")
print("Unique row IDs:      ", f"{unique_row_ids:,}")
print("Missing lineage rows:", f"{bad_lineage_count:,}")
print("Invalid row numbers: ", f"{invalid_row_numbers:,}")
print("\nColumns:")
print(saved_df.columns)

In [0]:

from pyspark.sql import functions as F

# ============================================================
# RetailRecon | Phase 2
# Typed source layer + source overlap exclusions
# Development only; no production deployment
# ============================================================

RAW_PATH = (
    "/Volumes/portfolio_projects/retail_sales/raw/"
    "source_raw_parquet"
)

TYPED_PATH = (
    "/Volumes/portfolio_projects/retail_sales/raw/"
    "source_typed_parquet"
)

EXCLUSIONS_PATH = (
    "/Volumes/portfolio_projects/retail_sales/raw/"
    "source_overlap_exclusions_parquet"
)

EXPECTED_TOTAL_ROWS = 1_067_371

FIRST_SHEET = "Year 2009-2010"
SECOND_SHEET = "Year 2010-2011"

CUTOFF = F.to_timestamp(
    F.lit("2010-12-01 00:00:00")
)

# ------------------------------------------------------------
# 1. Read validated raw Parquet
# ------------------------------------------------------------

raw = spark.read.parquet(RAW_PATH)

required_columns = {
    "invoice",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "price",
    "customer_id",
    "country",
    "source_sheet",
    "source_row_number",
    "source_workbook_sha256",
    "row_id",
}

missing = required_columns - set(raw.columns)

if missing:
    raise ValueError(
        f"Raw source is missing columns: {sorted(missing)}"
    )

source_count = raw.count()

if source_count != EXPECTED_TOTAL_ROWS:
    raise ValueError(
        f"Raw row-count mismatch: {source_count:,}"
    )

print(f"Raw source verified: {source_count:,} rows")


# ------------------------------------------------------------
# 2. Preserve original values before casting
# ------------------------------------------------------------

prepared = (
    raw
    .withColumn("invoice_raw", F.col("invoice"))
    .withColumn("stock_code_raw", F.col("stock_code"))
    .withColumn("quantity_raw", F.col("quantity"))
    .withColumn("invoice_date_raw", F.col("invoice_date"))
    .withColumn("price_raw", F.col("price"))
    .withColumn("customer_id_raw", F.col("customer_id"))
)


# ------------------------------------------------------------
# 3. Normalize identifiers and parse typed fields
# ------------------------------------------------------------

# Invoice and stock codes remain strings.
prepared = (
    prepared
    .withColumn(
        "invoice",
        F.when(
            F.trim(F.col("invoice_raw")) == "",
            F.lit(None).cast("string"),
        ).otherwise(F.trim(F.col("invoice_raw"))),
    )
    .withColumn(
        "stock_code",
        F.when(
            F.trim(F.col("stock_code_raw")) == "",
            F.lit(None).cast("string"),
        ).otherwise(F.trim(F.col("stock_code_raw"))),
    )
    .withColumn(
        "description",
        F.when(
            F.trim(F.col("description")) == "",
            F.lit(None).cast("string"),
        ).otherwise(F.trim(F.col("description"))),
    )
    .withColumn(
        "country",
        F.when(
            F.trim(F.col("country")) == "",
            F.lit(None).cast("string"),
        ).otherwise(F.trim(F.col("country"))),
    )
)

# Customer IDs remain strings, never numeric.
# Normalize Excel values such as 17850.0 to 17850 only when
# the complete value is a numeric identifier ending in .0.
prepared = prepared.withColumn(
    "customer_id",
    F.when(
        F.trim(F.col("customer_id_raw")) == "",
        F.lit(None).cast("string"),
    )
    .when(
        F.trim(F.col("customer_id_raw")).rlike(r"^[0-9]+\.0+$"),
        F.regexp_replace(
            F.trim(F.col("customer_id_raw")),
            r"\.0+$",
            "",
        ),
    )
    .otherwise(F.trim(F.col("customer_id_raw"))),
)

# Cast safely: malformed values become NULL and can be routed
# to exceptions in the following processing stage.
prepared = (
    prepared
    .withColumn(
        "quantity",
        F.expr(
            "try_cast(NULLIF(TRIM(quantity_raw), '') AS INT)"
        ),
    )
    .withColumn(
        "invoice_date",
        F.expr(
            "try_cast(NULLIF(TRIM(invoice_date_raw), '') AS TIMESTAMP)"
        ),
    )
    .withColumn(
        "price",
        F.expr(
            "try_cast(NULLIF(TRIM(price_raw), '') AS DECIMAL(18,4))"
        ),
    )
)

print("Typed columns created.")
print("Original raw values retained in *_raw columns.")


# ------------------------------------------------------------
# 4. Apply the explicit source-precedence rule
# ------------------------------------------------------------

# First worksheet: retain transactions strictly before cutoff.
# Second worksheet: retain transactions on or after cutoff.
#
# Rows with an unparseable/missing date are NOT classified as
# overlap exclusions. They remain in the typed source so they
# can be classified as date exceptions in the next phase.

first_sheet_exclusion = (
    (F.col("source_sheet") == FIRST_SHEET)
    & F.col("invoice_date").isNotNull()
    & (F.col("invoice_date") >= CUTOFF)
)

second_sheet_exclusion = (
    (F.col("source_sheet") == SECOND_SHEET)
    & F.col("invoice_date").isNotNull()
    & (F.col("invoice_date") < CUTOFF)
)

prepared = (
    prepared
    .withColumn(
        "exclusion_reason",
        F.when(
            first_sheet_exclusion,
            F.lit("FIRST_SHEET_ON_OR_AFTER_2010_12_01"),
        )
        .when(
            second_sheet_exclusion,
            F.lit("SECOND_SHEET_BEFORE_2010_12_01"),
        )
    )
    .withColumn(
        "_is_source_exclusion",
        F.col("exclusion_reason").isNotNull(),
    )
)

# Excluded rows are saved separately for auditability.
source_exclusions = (
    prepared
    .filter(F.col("_is_source_exclusion"))
    .drop("_is_source_exclusion")
    .withColumn(
        "exclusion_stage",
        F.lit("SOURCE_OVERLAP"),
    )
)

# All other rows proceed to data-quality and business rules.
typed_source = (
    prepared
    .filter(~F.col("_is_source_exclusion"))
    .drop("_is_source_exclusion")
)


# ------------------------------------------------------------
# 5. Validate row accounting before writing
# ------------------------------------------------------------

retained_count = typed_source.count()
exclusion_count = source_exclusions.count()

print("\nSource-precedence accounting")
print("-" * 45)
print(f"Raw source rows:    {source_count:,}")
print(f"Retained rows:      {retained_count:,}")
print(f"Excluded rows:      {exclusion_count:,}")

if retained_count + exclusion_count != source_count:
    raise ValueError(
        "Row-accounting failure: retained + excluded "
        "does not equal source rows."
    )

# IDs must remain unique across the two output datasets.
combined_ids = (
    typed_source.select("row_id")
    .unionByName(source_exclusions.select("row_id"))
)

unique_ids = combined_ids.distinct().count()

if unique_ids != source_count:
    raise ValueError(
        "Source row IDs are not unique across the outputs."
    )

print("PASS: All source rows are accounted for.")
print("PASS: Source row IDs remain unique.")


# ------------------------------------------------------------
# 6. Show typed-source schema
# ------------------------------------------------------------

print("\nTyped source schema:")
typed_source.printSchema()

print("\nExclusions by reason:")
source_exclusions.groupBy("exclusion_reason").count().show(
    truncate=False
)


# ------------------------------------------------------------
# 7. Write development outputs
# ------------------------------------------------------------

# These are dedicated output folders and may be rebuilt on rerun.
# The original raw Parquet source remains unchanged.

print("\nWriting typed source Parquet...")

(
    typed_source.write
    .format("parquet")
    .mode("overwrite")
    .option("compression", "snappy")
    .save(TYPED_PATH)
)

print("Typed source saved:", TYPED_PATH)

print("\nWriting source-overlap exclusions...")

(
    source_exclusions.write
    .format("parquet")
    .mode("overwrite")
    .option("compression", "snappy")
    .save(EXCLUSIONS_PATH)
)

print("Exclusions saved:", EXCLUSIONS_PATH)

print("\n" + "=" * 60)
print("SUCCESS: PHASE 2 SOURCE TRANSFORMATION COMPLETED")
print("=" * 60)
print(f"Raw rows:       {source_count:,}")
print(f"Retained rows:  {retained_count:,}")
print(f"Excluded rows:  {exclusion_count:,}")
print(f"Total accounted:{retained_count + exclusion_count:>11,}")

In [0]:

from pyspark.sql import functions as F

TYPED_PATH = (
    "/Volumes/portfolio_projects/retail_sales/raw/"
    "source_typed_parquet"
)

EXCLUSIONS_PATH = (
    "/Volumes/portfolio_projects/retail_sales/raw/"
    "source_overlap_exclusions_parquet"
)

EXPECTED_TOTAL_ROWS = 1_067_371

typed = spark.read.parquet(TYPED_PATH)
exclusions = spark.read.parquet(EXCLUSIONS_PATH)

retained_count = typed.count()
excluded_count = exclusions.count()

# Check the output accounting.
assert retained_count + excluded_count == EXPECTED_TOTAL_ROWS, (
    "Persisted row counts do not reconcile to the source."
)

# Check the actual Spark data types.
expected_types = {
    "invoice": "string",
    "stock_code": "string",
    "quantity": "int",
    "invoice_date": "timestamp",
    "price": "decimal(18,4)",
    "customer_id": "string",
}

actual_types = {
    field.name: field.dataType.simpleString()
    for field in typed.schema.fields
}

for column_name, expected_type in expected_types.items():
    actual_type = actual_types.get(column_name)

    print(
        f"{column_name}: {actual_type} "
        f"(expected {expected_type})"
    )

    assert actual_type == expected_type, (
        f"Unexpected type for {column_name}: {actual_type}"
    )

# Check that no excluded row is also in the retained dataset.
overlap_ids = (
    typed.select("row_id")
    .join(
        exclusions.select("row_id"),
        on="row_id",
        how="inner",
    )
)

assert overlap_ids.limit(1).count() == 0, (
    "A source row appears in both outputs."
)

# Reconcile row IDs between the original source and outputs.
raw = spark.read.parquet(
    "/Volumes/portfolio_projects/retail_sales/raw/"
    "source_raw_parquet"
)

output_ids = (
    typed.select("row_id")
    .unionByName(exclusions.select("row_id"))
)

missing_ids = (
    raw.select("row_id")
    .join(output_ids, on="row_id", how="left_anti")
    .limit(1)
    .count()
)

unexpected_ids = (
    output_ids
    .join(raw.select("row_id"), on="row_id", how="left_anti")
    .limit(1)
    .count()
)

assert missing_ids == 0, "Some source row IDs are missing."
assert unexpected_ids == 0, "Unexpected source row IDs found."

print("\n" + "=" * 55)
print("PASS: PERSISTED SOURCE VALIDATION")
print("=" * 55)
print(f"Retained source rows: {retained_count:,}")
print(f"Overlap exclusions:   {excluded_count:,}")
print(f"Total accounted:      {retained_count + excluded_count:,}")
print("Data types:           PASS")
print("Row-ID reconciliation: PASS")

In [0]:

from pyspark.sql import functions as F

# ============================================================
# RetailRecon | Phase 3: Transaction Classification
# Development only. No production resources.
# ============================================================

BASE_PATH = "/Volumes/portfolio_projects/retail_sales/raw"

TYPED_PATH = f"{BASE_PATH}/source_typed_parquet"

LEDGER_PATH = f"{BASE_PATH}/classified_sales_ledger"
ADJUSTMENTS_PATH = f"{BASE_PATH}/classified_adjustments"
EXCEPTIONS_PATH = f"{BASE_PATH}/classified_exceptions"

ADJUSTMENT_CODES = [
    "POST", "DOT", "D", "M", "C2", "B", "AMAZONFEE"
]

# ------------------------------------------------------------
# 1. Read and verify the typed source
# ------------------------------------------------------------

typed = spark.read.parquet(TYPED_PATH)

source_count = typed.count()

print(f"Typed source rows: {source_count:,}")

required_columns = {
    "row_id", "invoice", "stock_code", "quantity",
    "invoice_date", "price", "customer_id",
    "source_sheet", "source_row_number"
}

missing_columns = required_columns - set(typed.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

if typed.filter(F.col("row_id").isNull()).limit(1).count():
    raise ValueError("Null row_id found in typed source.")

# ------------------------------------------------------------
# 2. Assign data-quality reason codes
# ------------------------------------------------------------

# Multiple failures can be captured for the same transaction.
# Customer ID is intentionally not a required field.

core_reason_codes = F.concat_ws(
    "|",
    F.when(
        F.col("invoice").isNull()
        | (F.trim(F.col("invoice")) == ""),
        F.lit("MISSING_INVOICE")
    ),
    F.when(
        F.col("stock_code").isNull()
        | (F.trim(F.col("stock_code")) == ""),
        F.lit("MISSING_STOCK_CODE")
    ),
    F.when(
        F.col("invoice_date").isNull(),
        F.lit("MISSING_OR_INVALID_INVOICE_DATE")
    ),
    F.when(
        F.col("quantity").isNull(),
        F.lit("MISSING_OR_INVALID_QUANTITY")
    ),
    F.when(
        F.col("quantity") == 0,
        F.lit("ZERO_QUANTITY")
    ),
    F.when(
        F.col("price").isNull(),
        F.lit("MISSING_OR_INVALID_PRICE")
    ),
    F.when(
        F.col("price") <= 0,
        F.lit("NONPOSITIVE_PRICE")
    ),
)

prepared = typed.withColumn(
    "_core_reason_codes",
    core_reason_codes
)

# Invalid core records become exceptions.
data_quality_exceptions = (
    prepared
    .filter(F.length("_core_reason_codes") > 0)
    .withColumn(
        "exception_reason_codes",
        F.col("_core_reason_codes")
    )
    .withColumn(
        "exception_type",
        F.lit("DATA_QUALITY")
    )
    .drop("_core_reason_codes")
)

# Only records that pass core checks proceed to classification.
valid = (
    prepared
    .filter(F.length("_core_reason_codes") == 0)
    .drop("_core_reason_codes")
)

# ------------------------------------------------------------
# 3. Separate non-merchandise adjustments
# ------------------------------------------------------------

normalized_stock_code = F.upper(F.trim(F.col("stock_code")))

is_adjustment = normalized_stock_code.isin(*ADJUSTMENT_CODES)

adjustments = (
    valid
    .filter(is_adjustment)
    .withColumn(
        "adjustment_code",
        normalized_stock_code
    )
    .withColumn(
        "record_type",
        F.lit("ADJUSTMENT")
    )
)

# Merchandise classification excludes all adjustment codes.
merchandise = valid.filter(~is_adjustment)

# ------------------------------------------------------------
# 4. Classify merchandise sales and cancellations
# ------------------------------------------------------------

is_c_invoice = F.upper(F.col("invoice")).startswith("C")

classified = merchandise.withColumn(
    "transaction_type",
    F.when(
        (F.col("quantity") > 0) & (~is_c_invoice),
        F.lit("SALE")
    )
    .when(
        (F.col("quantity") < 0) & is_c_invoice,
        F.lit("CANCELLATION")
    )
)

# A positive-quantity C-prefixed invoice is inconsistent.
# A negative-quantity non-C-prefixed invoice is inconsistent.
prefix_mismatch_reason = (
    F.when(
        (F.col("quantity") > 0) & is_c_invoice,
        F.lit("POSITIVE_QUANTITY_C_PREFIX_INVOICE")
    )
    .when(
        (F.col("quantity") < 0) & (~is_c_invoice),
        F.lit("NEGATIVE_QUANTITY_NON_C_PREFIX_INVOICE")
    )
)

classification_exceptions = (
    classified
    .filter(F.col("transaction_type").isNull())
    .withColumn(
        "exception_reason_codes",
        prefix_mismatch_reason
    )
    .withColumn(
        "exception_type",
        F.lit("CLASSIFICATION")
    )
    .drop("transaction_type")
)

# ------------------------------------------------------------
# 5. Build the merchandise sales ledger
# ------------------------------------------------------------

# Quantity * unit price gives the signed extended line value.
# Cancellation values are negative; reports can use their
# absolute values when calculating cancellation amounts.

sales_ledger = (
    classified
    .filter(F.col("transaction_type").isNotNull())
    .withColumn(
        "signed_line_value",
        (
            F.col("quantity").cast("decimal(10,0)")
            * F.col("price")
        ).cast("decimal(30,4)")
    )
    .withColumn(
        "invoice_year",
        F.year("invoice_date")
    )
    .withColumn(
        "invoice_month",
        F.month("invoice_date")
    )
)

# Combine both types of exceptions into one auditable output.
exceptions = data_quality_exceptions.unionByName(
    classification_exceptions,
    allowMissingColumns=True
)

# ------------------------------------------------------------
# 6. Validate mutually exclusive row accounting
# ------------------------------------------------------------

ledger_count = sales_ledger.count()
adjustment_count = adjustments.count()
exception_count = exceptions.count()

classified_total = (
    ledger_count + adjustment_count + exception_count
)

print("\nClassification counts")
print("-" * 50)
print(f"Typed source:       {source_count:,}")
print(f"Sales ledger:       {ledger_count:,}")
print(f"Adjustments:        {adjustment_count:,}")
print(f"Exceptions:         {exception_count:,}")
print(f"Classified total:   {classified_total:,}")

if classified_total != source_count:
    raise ValueError(
        "Row accounting failed: output counts do not equal "
        "the typed source count."
    )

# Check that each output has unique row IDs.
for output_name, output_df in [
    ("sales_ledger", sales_ledger),
    ("adjustments", adjustments),
    ("exceptions", exceptions),
]:
    output_rows = output_df.count()
    unique_ids = output_df.select("row_id").distinct().count()

    if output_rows != unique_ids:
        raise ValueError(
            f"{output_name} contains duplicate row IDs: "
            f"{output_rows:,} rows vs {unique_ids:,} unique IDs."
        )

# Check uniqueness across all three output datasets.
all_output_ids = (
    sales_ledger.select("row_id")
    .unionByName(adjustments.select("row_id"))
    .unionByName(exceptions.select("row_id"))
)

unique_output_ids = all_output_ids.distinct().count()

if unique_output_ids != source_count:
    raise ValueError(
        "A source row was duplicated or appears in multiple outputs."
    )

print("\nPASS: Each retained row belongs to exactly one output.")

# ------------------------------------------------------------
# 7. Inspect results before writing
# ------------------------------------------------------------

print("\nMerchandise transaction types:")
sales_ledger.groupBy("transaction_type").count().show(
    truncate=False
)

print("\nAdjustments by code:")
adjustments.groupBy("adjustment_code").count().show(
    truncate=False
)

print("\nExceptions by type and reason:")
exceptions.groupBy(
    "exception_type",
    "exception_reason_codes"
).count().orderBy(
    F.desc("count")
).show(50, truncate=False)

# ------------------------------------------------------------
# 8. Write separate development Parquet outputs
# ------------------------------------------------------------

# These paths are dedicated generated outputs.
# The original workbook and previous raw/typed layers are untouched.

print("\nWriting sales ledger...")

(
    sales_ledger.write
    .format("parquet")
    .mode("overwrite")
    .option("compression", "snappy")
    .partitionBy("invoice_year", "invoice_month")
    .save(LEDGER_PATH)
)

print("Sales ledger saved:", LEDGER_PATH)

print("\nWriting adjustments...")

(
    adjustments.write
    .format("parquet")
    .mode("overwrite")
    .option("compression", "snappy")
    .save(ADJUSTMENTS_PATH)
)

print("Adjustments saved:", ADJUSTMENTS_PATH)

print("\nWriting exceptions...")

(
    exceptions.write
    .format("parquet")
    .mode("overwrite")
    .option("compression", "snappy")
    .save(EXCEPTIONS_PATH)
)

print("Exceptions saved:", EXCEPTIONS_PATH)

print("\n" + "=" * 60)
print("SUCCESS: PHASE 3 CLASSIFICATION COMPLETED")
print("=" * 60)
print(f"Sales ledger: {ledger_count:,}")
print(f"Adjustments:  {adjustment_count:,}")
print(f"Exceptions:   {exception_count:,}")
print(f"Total:        {classified_total:,}")

In [0]:

from collections import deque
from decimal import Decimal
from hashlib import sha256

import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType,
    LongType,
    IntegerType,
)

# ============================================================
# RetailRecon | Phase 4: Corrected FIFO Allocation
# Development only. No production deployment.
# ============================================================

BASE_PATH = "/Volumes/portfolio_projects/retail_sales/raw"

LEDGER_PATH = f"{BASE_PATH}/classified_sales_ledger"
ALLOCATIONS_PATH = f"{BASE_PATH}/cancellation_allocations_parquet"
STATUS_PATH = f"{BASE_PATH}/cancellation_status_parquet"
STAGING_PATH = f"{BASE_PATH}/_fifo_staging"

ledger = spark.read.parquet(LEDGER_PATH)

sales = ledger.filter(F.col("transaction_type") == "SALE")
cancellations = ledger.filter(
    F.col("transaction_type") == "CANCELLATION"
)

cancellation_count = cancellations.count()

print("Sale rows:", f"{sales.count():,}")
print("Cancellation rows:", f"{cancellation_count:,}")

# ------------------------------------------------------------
# 1. Define the common output schema
# ------------------------------------------------------------

output_schema = StructType([
    StructField("record_type", StringType(), True),
    StructField("allocation_id", StringType(), True),
    StructField("cancellation_row_id", StringType(), True),
    StructField("sale_row_id", StringType(), True),
    StructField("cancellation_invoice", StringType(), True),
    StructField("sale_invoice", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("sale_stock_code", StringType(), True),
    StructField("cancellation_stock_code", StringType(), True),
    StructField("unit_price", DecimalType(18, 4), True),
    StructField("sale_date", TimestampType(), True),
    StructField("cancellation_date", TimestampType(), True),
    StructField("requested_quantity", LongType(), True),
    StructField("matched_quantity", LongType(), True),
    StructField("unmatched_quantity", LongType(), True),
    StructField("allocated_quantity", LongType(), True),
    StructField("allocation_value", DecimalType(30, 4), True),
    StructField("days_to_cancellation", IntegerType(), True),
    StructField("cancellation_status", StringType(), True),
    StructField("unmatched_reason", StringType(), True),
    StructField("eligible_prior_sales_quantity", LongType(), True),
])

OUTPUT_COLUMNS = [field.name for field in output_schema.fields]

INTEGER_COLUMNS = [
    "requested_quantity",
    "matched_quantity",
    "unmatched_quantity",
    "allocated_quantity",
    "days_to_cancellation",
    "eligible_prior_sales_quantity",
]

TIMESTAMP_COLUMNS = ["sale_date", "cancellation_date"]


# ------------------------------------------------------------
# 2. FIFO algorithm applied independently to each matching key
# ------------------------------------------------------------

def process_fifo_group(pdf):
    """
    Match cancellations to earlier sales using a FIFO queue.

    Grouping keys:
      customer_id + normalized stock_code + unit_price

    Equal timestamps:
      cancellations are processed before sales.

    Unmatched cancellation quantity is never carried forward
    to consume future purchases.
    """

    if pdf.empty:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    pdf = pdf.copy()

    pdf["_event_priority"] = pdf[
        "transaction_type"
    ].map({
        "CANCELLATION": 0,
        "SALE": 1,
    })

    pdf = pdf.sort_values(
        by=[
            "invoice_date",
            "_event_priority",
            "invoice",
            "row_id",
        ],
        ascending=True,
        kind="mergesort",
    )

    # Each lot records a sale and its remaining quantity.
    inventory = deque()
    records = []

    # Total quantity ever purchased strictly before the current
    # timestamp. This is used to explain unmatched cancellations.
    prior_sales_quantity = 0

    for event in pdf.to_dict("records"):

        event_type = event["transaction_type"]

        # --------------------------
        # Sales add inventory.
        # --------------------------
        if event_type == "SALE":

            quantity = int(event["quantity"])

            if quantity <= 0:
                raise ValueError(
                    f"Invalid sale quantity for row {event['row_id']}"
                )

            inventory.append({
                "row_id": event["row_id"],
                "invoice": event["invoice"],
                "invoice_date": event["invoice_date"],
                "stock_code": event["stock_code"],
                "price": event["price"],
                "remaining_quantity": quantity,
            })

            prior_sales_quantity += quantity
            continue

        # --------------------------
        # Cancellations consume only
        # inventory already in the FIFO.
        # --------------------------
        requested = abs(int(event["quantity"]))

        if requested <= 0:
            raise ValueError(
                f"Invalid cancellation quantity for row {event['row_id']}"
            )

        remaining_request = requested
        matched = 0

        while remaining_request > 0 and inventory:

            # Remove fully consumed lots.
            while (
                inventory
                and inventory[0]["remaining_quantity"] == 0
            ):
                inventory.popleft()

            if not inventory:
                break

            lot = inventory[0]

            allocated = min(
                remaining_request,
                lot["remaining_quantity"],
            )

            if allocated <= 0:
                raise ValueError("Non-positive FIFO allocation detected.")

            price = Decimal(str(event["price"]))
            allocation_value = (
                price * Decimal(allocated)
            ).quantize(Decimal("0.0001"))

            sale_date = pd.Timestamp(lot["invoice_date"])
            cancellation_date = pd.Timestamp(event["invoice_date"])

            # The chronological event order should guarantee this.
            if sale_date >= cancellation_date:
                raise ValueError(
                    "FIFO attempted to match a sale at or after "
                    f"cancellation {event['row_id']}"
                )

            pair_key = (
                f"{lot['row_id']}||{event['row_id']}"
            )

            records.append({
                "record_type": "ALLOCATION",
                "allocation_id": sha256(
                    pair_key.encode("utf-8")
                ).hexdigest(),
                "cancellation_row_id": event["row_id"],
                "sale_row_id": lot["row_id"],
                "cancellation_invoice": event["invoice"],
                "sale_invoice": lot["invoice"],
                "customer_id": event["customer_id"],
                "sale_stock_code": lot["stock_code"],
                "cancellation_stock_code": event["stock_code"],
                "unit_price": price,
                "sale_date": sale_date.to_pydatetime(),
                "cancellation_date": cancellation_date.to_pydatetime(),
                "requested_quantity": None,
                "matched_quantity": None,
                "unmatched_quantity": None,
                "allocated_quantity": allocated,
                "allocation_value": allocation_value,
                "days_to_cancellation": (
                    cancellation_date - sale_date
                ).days,
                "cancellation_status": None,
                "unmatched_reason": None,
                "eligible_prior_sales_quantity": None,
            })

            # Consume this sale lot exactly once.
            lot["remaining_quantity"] -= allocated
            remaining_request -= allocated
            matched += allocated

        unmatched = requested - matched

        if unmatched == 0:
            status = "MATCHED"
            reason = None

        elif matched > 0:
            status = "PARTIALLY_MATCHED"
            reason = "INSUFFICIENT_PRIOR_QUANTITY"

        elif prior_sales_quantity == 0:
            status = "UNMATCHED"
            reason = "NO_PRIOR_ELIGIBLE_SALE"

        else:
            status = "UNMATCHED"
            reason = "NO_AVAILABLE_PRIOR_QUANTITY"

        records.append({
            "record_type": "STATUS",
            "allocation_id": None,
            "cancellation_row_id": event["row_id"],
            "sale_row_id": None,
            "cancellation_invoice": event["invoice"],
            "sale_invoice": None,
            "customer_id": event["customer_id"],
            "sale_stock_code": None,
            "cancellation_stock_code": event["stock_code"],
            "unit_price": Decimal(str(event["price"])),
            "sale_date": None,
            "cancellation_date": pd.Timestamp(
                event["invoice_date"]
            ).to_pydatetime(),
            "requested_quantity": requested,
            "matched_quantity": matched,
            "unmatched_quantity": unmatched,
            "allocated_quantity": None,
            "allocation_value": None,
            "days_to_cancellation": None,
            "cancellation_status": status,
            "unmatched_reason": reason,
            "eligible_prior_sales_quantity": prior_sales_quantity,
        })

    result = pd.DataFrame(records, columns=OUTPUT_COLUMNS)

    # Preserve integer and timestamp types for Arrow/Spark output.
    for column in INTEGER_COLUMNS:
        result[column] = pd.array(
            result[column],
            dtype="Int64",
        )

    for column in TIMESTAMP_COLUMNS:
        result[column] = pd.to_datetime(
            result[column],
            errors="coerce",
        )

    return result


# ------------------------------------------------------------
# 3. Prepare eligible non-null customer events
# ------------------------------------------------------------

matchable_events = (
    ledger
    .filter(F.col("customer_id").isNotNull())
    .withColumn(
        "_match_stock_code",
        F.upper(F.trim(F.col("stock_code"))),
    )
    .select(
        "row_id",
        "invoice",
        "invoice_date",
        "customer_id",
        "stock_code",
        "_match_stock_code",
        "price",
        "quantity",
        "transaction_type",
    )
)

# A customer must not match their own missing ID or another
# missing ID. Those cancellation rows receive explicit status
# records below rather than being passed to the FIFO algorithm.

grouped_results = (
    matchable_events
    .groupBy(
        "customer_id",
        "_match_stock_code",
        "price",
    )
    .applyInPandas(
        process_fifo_group,
        schema=output_schema,
    )
)

# ------------------------------------------------------------
# 4. Create explicit unmatched statuses for null customers
# ------------------------------------------------------------

null_customer_status = (
    cancellations
    .filter(F.col("customer_id").isNull())
    .select(
        F.lit("STATUS").alias("record_type"),
        F.lit(None).cast("string").alias("allocation_id"),
        F.col("row_id").alias("cancellation_row_id"),
        F.lit(None).cast("string").alias("sale_row_id"),
        F.col("invoice").alias("cancellation_invoice"),
        F.lit(None).cast("string").alias("sale_invoice"),
        F.col("customer_id").cast("string").alias("customer_id"),
        F.lit(None).cast("string").alias("sale_stock_code"),
        F.col("stock_code").alias("cancellation_stock_code"),
        F.col("price").cast("decimal(18,4)").alias("unit_price"),
        F.lit(None).cast("timestamp").alias("sale_date"),
        F.col("invoice_date").cast("timestamp").alias(
            "cancellation_date"
        ),
        F.abs(F.col("quantity").cast("long")).alias(
            "requested_quantity"
        ),
        F.lit(0).cast("long").alias("matched_quantity"),
        F.abs(F.col("quantity").cast("long")).alias(
            "unmatched_quantity"
        ),
        F.lit(None).cast("long").alias("allocated_quantity"),
        F.lit(None).cast("decimal(30,4)").alias(
            "allocation_value"
        ),
        F.lit(None).cast("int").alias("days_to_cancellation"),
        F.lit("UNMATCHED").alias("cancellation_status"),
        F.lit("MISSING_CUSTOMER_ID").alias("unmatched_reason"),
        F.lit(0).cast("long").alias(
            "eligible_prior_sales_quantity"
        ),
    )
)

all_results_raw = grouped_results.unionByName(null_customer_status)

# Materialize once to a staging Parquet location so that
# subsequent reads, checks, and writes reuse this result.
# .persist() is not supported on Serverless compute.
(
    all_results_raw.write
    .format("parquet")
    .mode("overwrite")
    .option("compression", "snappy")
    .save(STAGING_PATH)
)

all_results = spark.read.parquet(STAGING_PATH)

# Materialize once so that checks and writes reuse this result.
result_rows = all_results.count()

allocations = all_results.filter(
    F.col("record_type") == "ALLOCATION"
).select(
    "allocation_id",
    "cancellation_row_id",
    "sale_row_id",
    "cancellation_invoice",
    "sale_invoice",
    "customer_id",
    "sale_stock_code",
    "cancellation_stock_code",
    "unit_price",
    "sale_date",
    "cancellation_date",
    "allocated_quantity",
    "allocation_value",
    "days_to_cancellation",
)

status = all_results.filter(
    F.col("record_type") == "STATUS"
).select(
    "cancellation_row_id",
    "cancellation_invoice",
    "cancellation_date",
    "customer_id",
    "cancellation_stock_code",
    "unit_price",
    "requested_quantity",
    "matched_quantity",
    "unmatched_quantity",
    "cancellation_status",
    "unmatched_reason",
    "eligible_prior_sales_quantity",
)

# ------------------------------------------------------------
# 5. Validate the allocation before saving
# ------------------------------------------------------------

status_count = status.count()
allocation_count = allocations.count()

if status_count != cancellation_count:
    raise ValueError(
        f"Expected {cancellation_count:,} cancellation statuses; "
        f"found {status_count:,}"
    )

if status.select("cancellation_row_id").distinct().count() != status_count:
    raise ValueError("Duplicate cancellation status IDs detected.")

if allocations.select("allocation_id").distinct().count() != allocation_count:
    raise ValueError("Duplicate allocation IDs detected.")

# Every cancellation must reconcile:
# requested = matched + unmatched.
bad_status = status.filter(
    F.col("requested_quantity")
    != F.col("matched_quantity") + F.col("unmatched_quantity")
)

if bad_status.limit(1).count() > 0:
    raise ValueError("Cancellation quantity reconciliation failed.")

if status.filter(
    (F.col("matched_quantity") > F.col("requested_quantity"))
    | (F.col("unmatched_quantity") < 0)
).limit(1).count() > 0:
    raise ValueError("Cancellation quantity over-allocation detected.")

# Allocation timestamps must be strictly earlier.
if allocations.filter(
    F.col("sale_date") >= F.col("cancellation_date")
).limit(1).count() > 0:
    raise ValueError("Same-time or future-sale allocation detected.")

# Every sold unit can be allocated at most once.
sale_consumption = (
    allocations
    .groupBy("sale_row_id")
    .agg(
        F.sum("allocated_quantity").alias("consumed_quantity")
    )
)

sale_quantities = sales.select(
    F.col("row_id").alias("sale_row_id"),
    F.col("quantity").cast("long").alias("sold_quantity"),
)

overallocated_sales = (
    sale_consumption
    .join(sale_quantities, on="sale_row_id", how="inner")
    .filter(
        F.col("consumed_quantity") > F.col("sold_quantity")
    )
)

if overallocated_sales.limit(1).count() > 0:
    raise ValueError("A sale was allocated beyond its sold quantity.")

allocated_units = (
    allocations
    .agg(
        F.coalesce(
            F.sum("allocated_quantity"),
            F.lit(0).cast("long"),
        ).alias("total")
    )
    .first()["total"]
)

matched_units = (
    status
    .agg(
        F.coalesce(
            F.sum("matched_quantity"),
            F.lit(0).cast("long"),
        ).alias("total")
    )
    .first()["total"]
)

if allocated_units != matched_units:
    raise ValueError(
        f"Allocation/status mismatch: "
        f"{allocated_units} != {matched_units}"
    )

print("\nFIFO validation passed.")
print(f"Cancellation statuses: {status_count:,}")
print(f"Allocation rows:       {allocation_count:,}")
print(f"Allocated units:        {allocated_units:,}")

# ------------------------------------------------------------
# 6. Save both development outputs
# ------------------------------------------------------------

(
    allocations.write
    .format("parquet")
    .mode("overwrite")
    .option("compression", "snappy")
    .save(ALLOCATIONS_PATH)
)

(
    status.write
    .format("parquet")
    .mode("overwrite")
    .option("compression", "snappy")
    .save(STATUS_PATH)
)

print("\nSaved allocations:", ALLOCATIONS_PATH)
print("Saved status:     ", STATUS_PATH)

print("\nCancellation status summary:")
status.groupBy(
    "cancellation_status",
    "unmatched_reason",
).agg(
    F.count("*").alias("cancellation_rows"),
    F.sum("requested_quantity").alias("requested_units"),
    F.sum("matched_quantity").alias("matched_units"),
    F.sum("unmatched_quantity").alias("unmatched_units"),
).orderBy(
    "cancellation_status",
    "unmatched_reason",
).show(100, truncate=False)

print("\nSUCCESS: FIFO allocation and validation completed.")

# Clean up the staging directory after successful validation.
dbutils.fs.rm(STAGING_PATH, recurse=True)print("Staging directory cleaned up:", STAGING_PATH)

In [0]:
# ============================================================
# Synthetic FIFO Tests — Serverless Compatibility Verification
# Self-contained: independent of the retail dataset.
# Verifies applyInPandas works on Serverless compute.
# ============================================================

from collections import deque
from datetime import datetime
from decimal import Decimal
from hashlib import sha256

import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType,
    LongType,
    IntegerType,
)

# ------------------------------------------------------------
# Output schema (identical to cell 10)
# ------------------------------------------------------------

_test_schema = StructType([
    StructField("record_type", StringType(), True),
    StructField("allocation_id", StringType(), True),
    StructField("cancellation_row_id", StringType(), True),
    StructField("sale_row_id", StringType(), True),
    StructField("cancellation_invoice", StringType(), True),
    StructField("sale_invoice", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("sale_stock_code", StringType(), True),
    StructField("cancellation_stock_code", StringType(), True),
    StructField("unit_price", DecimalType(18, 4), True),
    StructField("sale_date", TimestampType(), True),
    StructField("cancellation_date", TimestampType(), True),
    StructField("requested_quantity", LongType(), True),
    StructField("matched_quantity", LongType(), True),
    StructField("unmatched_quantity", LongType(), True),
    StructField("allocated_quantity", LongType(), True),
    StructField("allocation_value", DecimalType(30, 4), True),
    StructField("days_to_cancellation", IntegerType(), True),
    StructField("cancellation_status", StringType(), True),
    StructField("unmatched_reason", StringType(), True),
    StructField("eligible_prior_sales_quantity", LongType(), True),
])

_test_cols = [f.name for f in _test_schema.fields]
_test_int_cols = [
    "requested_quantity", "matched_quantity",
    "unmatched_quantity", "allocated_quantity",
    "days_to_cancellation",
    "eligible_prior_sales_quantity",
]
_test_ts_cols = ["sale_date", "cancellation_date"]

# ------------------------------------------------------------
# FIFO function (identical algorithm to cell 10)
# ------------------------------------------------------------

def _fifo_group(pdf):
    if pdf.empty:
        return pd.DataFrame(columns=_test_cols)

    pdf = pdf.copy()
    pdf["_event_priority"] = pdf[
        "transaction_type"
    ].map({"CANCELLATION": 0, "SALE": 1})
    pdf = pdf.sort_values(
        by=[
            "invoice_date", "_event_priority",
            "invoice", "row_id",
        ],
        ascending=True, kind="mergesort",
    )

    inventory = deque()
    records = []
    prior_sales_quantity = 0

    for event in pdf.to_dict("records"):
        event_type = event["transaction_type"]

        if event_type == "SALE":
            quantity = int(event["quantity"])
            if quantity <= 0:
                raise ValueError(
                    f"Invalid sale quantity for row {event['row_id']}"
                )
            inventory.append({
                "row_id": event["row_id"],
                "invoice": event["invoice"],
                "invoice_date": event["invoice_date"],
                "stock_code": event["stock_code"],
                "price": event["price"],
                "remaining_quantity": quantity,
            })
            prior_sales_quantity += quantity
            continue

        requested = abs(int(event["quantity"]))
        if requested <= 0:
            raise ValueError(
                f"Invalid cancellation quantity for row {event['row_id']}"
            )

        remaining_request = requested
        matched = 0

        while remaining_request > 0 and inventory:
            while (
                inventory
                and inventory[0]["remaining_quantity"] == 0
            ):
                inventory.popleft()
            if not inventory:
                break

            lot = inventory[0]
            allocated = min(
                remaining_request, lot["remaining_quantity"]
            )
            if allocated <= 0:
                raise ValueError(
                    "Non-positive FIFO allocation detected."
                )

            price = Decimal(str(event["price"]))
            allocation_value = (
                price * Decimal(allocated)
            ).quantize(Decimal("0.0001"))

            sale_date = pd.Timestamp(lot["invoice_date"])
            cancellation_date = pd.Timestamp(
                event["invoice_date"]
            )

            if sale_date >= cancellation_date:
                raise ValueError(
                    "FIFO attempted to match a sale at or after "
                    f"cancellation {event['row_id']}"
                )

            pair_key = f"{lot['row_id']}||{event['row_id']}"

            records.append({
                "record_type": "ALLOCATION",
                "allocation_id": sha256(
                    pair_key.encode("utf-8")
                ).hexdigest(),
                "cancellation_row_id": event["row_id"],
                "sale_row_id": lot["row_id"],
                "cancellation_invoice": event["invoice"],
                "sale_invoice": lot["invoice"],
                "customer_id": event["customer_id"],
                "sale_stock_code": lot["stock_code"],
                "cancellation_stock_code": event["stock_code"],
                "unit_price": price,
                "sale_date": sale_date.to_pydatetime(),
                "cancellation_date": cancellation_date.to_pydatetime(),
                "requested_quantity": None,
                "matched_quantity": None,
                "unmatched_quantity": None,
                "allocated_quantity": allocated,
                "allocation_value": allocation_value,
                "days_to_cancellation": (
                    cancellation_date - sale_date
                ).days,
                "cancellation_status": None,
                "unmatched_reason": None,
                "eligible_prior_sales_quantity": None,
            })

            lot["remaining_quantity"] -= allocated
            remaining_request -= allocated
            matched += allocated

        unmatched = requested - matched

        if unmatched == 0:
            status, reason = "MATCHED", None
        elif matched > 0:
            status, reason = (
                "PARTIALLY_MATCHED",
                "INSUFFICIENT_PRIOR_QUANTITY",
            )
        elif prior_sales_quantity == 0:
            status, reason = (
                "UNMATCHED", "NO_PRIOR_ELIGIBLE_SALE"
            )
        else:
            status, reason = (
                "UNMATCHED", "NO_AVAILABLE_PRIOR_QUANTITY"
            )

        records.append({
            "record_type": "STATUS",
            "allocation_id": None,
            "cancellation_row_id": event["row_id"],
            "sale_row_id": None,
            "cancellation_invoice": event["invoice"],
            "sale_invoice": None,
            "customer_id": event["customer_id"],
            "sale_stock_code": None,
            "cancellation_stock_code": event["stock_code"],
            "unit_price": Decimal(str(event["price"])),
            "sale_date": None,
            "cancellation_date": pd.Timestamp(
                event["invoice_date"]
            ).to_pydatetime(),
            "requested_quantity": requested,
            "matched_quantity": matched,
            "unmatched_quantity": unmatched,
            "allocated_quantity": None,
            "allocation_value": None,
            "days_to_cancellation": None,
            "cancellation_status": status,
            "unmatched_reason": reason,
            "eligible_prior_sales_quantity": prior_sales_quantity,
        })

    result = pd.DataFrame(records, columns=_test_cols)
    for column in _test_int_cols:
        result[column] = pd.array(result[column], dtype="Int64")
    for column in _test_ts_cols:
        result[column] = pd.to_datetime(
            result[column], errors="coerce"
        )
    return result


# ------------------------------------------------------------
# Test helpers
# ------------------------------------------------------------

_input_schema = StructType([
    StructField("row_id", StringType(), True),
    StructField("invoice", StringType(), True),
    StructField("invoice_date", TimestampType(), True),
    StructField("customer_id", StringType(), True),
    StructField("stock_code", StringType(), True),
    StructField("_match_stock_code", StringType(), True),
    StructField("price", DecimalType(18, 4), True),
    StructField("quantity", LongType(), True),
    StructField("transaction_type", StringType(), True),
])


def _make_events(rows):
    """Create a Spark DataFrame from a list of tuples."""
    return spark.createDataFrame(rows, _input_schema)


def _run_fifo(events_df):
    """Run FIFO and collect results as pandas DataFrame."""
    result = (
        events_df
        .groupBy("customer_id", "_match_stock_code", "price")
        .applyInPandas(_fifo_group, schema=_test_schema)
    )
    return result.toPandas()


def _allocs(pdf):
    return pdf[pdf["record_type"] == "ALLOCATION"]


def _stats(pdf):
    return pdf[pdf["record_type"] == "STATUS"]


# ------------------------------------------------------------
# Test A: Partial FIFO allocation
# ------------------------------------------------------------

print("=" * 60)
print("Test A: Partial FIFO allocation")
print("=" * 60)

events_a = _make_events([
    ("s1", "INV-S1", datetime(2024, 1, 1, 10, 0, 0),
     "CUST-A", "WIDGET", "WIDGET", Decimal("10.0000"), 5, "SALE"),
    ("c1", "INV-C1", datetime(2024, 1, 2, 10, 0, 0),
     "CUST-A", "WIDGET", "WIDGET", Decimal("10.0000"), -3, "CANCELLATION"),
    ("c2", "INV-C2", datetime(2024, 1, 3, 10, 0, 0),
     "CUST-A", "WIDGET", "WIDGET", Decimal("10.0000"), -4, "CANCELLATION"),
])

result_a = _run_fifo(events_a)
s_a = _stats(result_a)
a_a = _allocs(result_a)

assert len(s_a) == 2, f"Expected 2 statuses, got {len(s_a)}"

c1 = s_a[s_a["cancellation_row_id"] == "c1"].iloc[0]
c2 = s_a[s_a["cancellation_row_id"] == "c2"].iloc[0]

assert int(c1["matched_quantity"]) == 3 and int(c1["unmatched_quantity"]) == 0
assert int(c2["matched_quantity"]) == 2 and int(c2["unmatched_quantity"]) == 2
assert c1["cancellation_status"] == "MATCHED"
assert c2["cancellation_status"] == "PARTIALLY_MATCHED"

total_alloc = int(a_a["allocated_quantity"].sum())
assert total_alloc == 5, f"Total allocated: {total_alloc}, expected 5"

print("PASS: C1 matched=3/unmatched=0, C2 matched=2/unmatched=2")
print(f"      Total allocated = {total_alloc} (sale had 5)")

# ------------------------------------------------------------
# Test B: Cancellation before sale
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("Test B: Cancellation before sale")
print("=" * 60)

events_b = _make_events([
    ("c1", "INV-C1", datetime(2024, 1, 1, 10, 0, 0),
     "CUST-B", "WIDGET", "WIDGET", Decimal("10.0000"), -3, "CANCELLATION"),
    ("s1", "INV-S1", datetime(2024, 1, 2, 10, 0, 0),
     "CUST-B", "WIDGET", "WIDGET", Decimal("10.0000"), 5, "SALE"),
])

result_b = _run_fifo(events_b)
s_b = _stats(result_b)

assert len(s_b) == 1
s = s_b.iloc[0]
assert s["cancellation_status"] == "UNMATCHED"
assert s["unmatched_reason"] == "NO_PRIOR_ELIGIBLE_SALE"
assert int(s["matched_quantity"]) == 0
assert int(s["unmatched_quantity"]) == 3
assert len(_allocs(result_b)) == 0

print("PASS: Cancellation unmatched (NO_PRIOR_ELIGIBLE_SALE)")
print("      Later sale not consumed")

# ------------------------------------------------------------
# Test C: Same timestamp
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("Test C: Same timestamp")
print("=" * 60)

events_c = _make_events([
    ("s1", "INV-S1", datetime(2024, 1, 1, 10, 0, 0),
     "CUST-C", "WIDGET", "WIDGET", Decimal("10.0000"), 5, "SALE"),
    ("c1", "INV-C1", datetime(2024, 1, 1, 10, 0, 0),
     "CUST-C", "WIDGET", "WIDGET", Decimal("10.0000"), -3, "CANCELLATION"),
])

result_c = _run_fifo(events_c)
s_c = _stats(result_c)

assert len(s_c) == 1
s = s_c.iloc[0]
assert s["cancellation_status"] == "UNMATCHED"
assert s["unmatched_reason"] == "NO_PRIOR_ELIGIBLE_SALE"
assert int(s["matched_quantity"]) == 0
assert int(s["unmatched_quantity"]) == 3
assert len(_allocs(result_c)) == 0

print("PASS: Same-timestamp sale not consumed by cancellation")
print("      Cancellation processed before sale (priority 0 < 1)")

# ------------------------------------------------------------
# Test D: Null customer
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("Test D: Null customer")
print("=" * 60)

events_d = _make_events([
    ("s1", "INV-S1", datetime(2024, 1, 1, 10, 0, 0),
     None, "WIDGET", "WIDGET", Decimal("10.0000"), 5, "SALE"),
    ("c1", "INV-C1", datetime(2024, 1, 2, 10, 0, 0),
     None, "WIDGET", "WIDGET", Decimal("10.0000"), -3, "CANCELLATION"),
    ("c2", "INV-C2", datetime(2024, 1, 3, 10, 0, 0),
     None, "WIDGET", "WIDGET", Decimal("10.0000"), -2, "CANCELLATION"),
])

# Null customers are filtered out before applyInPandas (as in cell 10).
matchable_d = events_d.filter(F.col("customer_id").isNotNull())
result_d = _run_fifo(matchable_d)
assert len(result_d) == 0, (
    f"Expected no FIFO results for null customers, got {len(result_d)}"
)

null_cancels = events_d.filter(
    (F.col("customer_id").isNull())
    & (F.col("transaction_type") == "CANCELLATION")
)
assert null_cancels.count() == 2

print("PASS: Null customer records excluded from FIFO")
print("      No cross-matching between null-customer records")

# ------------------------------------------------------------
# Test E: Different matching keys
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("Test E: Different matching keys")
print("=" * 60)

events_e = _make_events([
    ("s1", "INV-S1", datetime(2024, 1, 1, 10, 0, 0),
     "CUST-E", "GADGET", "GADGET", Decimal("10.0000"), 5, "SALE"),
    ("s2", "INV-S2", datetime(2024, 1, 1, 10, 0, 0),
     "CUST-E", "GIZMO", "GIZMO", Decimal("10.0000"), 5, "SALE"),
    ("s3", "INV-S3", datetime(2024, 1, 1, 10, 0, 0),
     "CUST-E", "GADGET", "GADGET", Decimal("20.0000"), 5, "SALE"),
    ("c1", "INV-C1", datetime(2024, 1, 2, 10, 0, 0),
     "CUST-E", "GADGET", "GADGET", Decimal("10.0000"), -3, "CANCELLATION"),
    ("c2", "INV-C2", datetime(2024, 1, 2, 10, 0, 0),
     "CUST-E", "GIZMO", "GIZMO", Decimal("10.0000"), -3, "CANCELLATION"),
    ("c3", "INV-C3", datetime(2024, 1, 2, 10, 0, 0),
     "CUST-E", "GADGET", "GADGET", Decimal("20.0000"), -3, "CANCELLATION"),
])

result_e = _run_fifo(events_e)
s_e = _stats(result_e)
a_e = _allocs(result_e)

assert len(s_e) == 3, f"Expected 3 statuses, got {len(s_e)}"

for _, s in s_e.iterrows():
    assert s["cancellation_status"] == "MATCHED", (
        f"Expected MATCHED for {s['cancellation_row_id']}, "
        f"got {s['cancellation_status']}"
    )
    assert int(s["matched_quantity"]) == 3

# Verify allocations stay within their key groups.
for _, a in a_e.iterrows():
    assert a["sale_stock_code"] == a["cancellation_stock_code"], (
        f"Stock code mismatch: sale={a['sale_stock_code']}, "
        f"cancel={a['cancellation_stock_code']}"
    )

assert len(a_e) == 3

print("PASS: Cancellations matched only within same key group")
print("      Different stock codes and prices do not cross-match")

# ------------------------------------------------------------
# Test F: Rerun determinism
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("Test F: Rerun determinism")
print("=" * 60)

events_f = _make_events([
    ("s1", "INV-S1", datetime(2024, 1, 1, 10, 0, 0),
     "CUST-F", "WIDGET", "WIDGET", Decimal("10.0000"), 5, "SALE"),
    ("c1", "INV-C1", datetime(2024, 1, 2, 10, 0, 0),
     "CUST-F", "WIDGET", "WIDGET", Decimal("10.0000"), -3, "CANCELLATION"),
])

result_f1 = _run_fifo(events_f)
result_f2 = _run_fifo(events_f)

ids1 = sorted(
    result_f1[result_f1["record_type"] == "ALLOCATION"][
        "allocation_id"
    ].tolist()
)
ids2 = sorted(
    result_f2[result_f2["record_type"] == "ALLOCATION"][
        "allocation_id"
    ].tolist()
)

assert ids1 == ids2, "Allocation IDs differ between runs"
assert result_f1.equals(result_f2), "Results differ between runs"

print("PASS: Identical allocation IDs and results on rerun")
print("      Deterministic SHA-256 allocation IDs confirmed")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("ALL 6 SYNTHETIC TESTS PASSED")
print("applyInPandas confirmed working on Serverless compute.")
print("=" * 60)

In [0]:
# ============================================================
# Post-Write Validation: Read back saved outputs and verify
# ============================================================

from pyspark.sql import functions as F

ALLOCATIONS_PATH = f"{BASE_PATH}/cancellation_allocations_parquet"
STATUS_PATH = f"{BASE_PATH}/cancellation_status_parquet"
LEDGER_PATH = f"{BASE_PATH}/classified_sales_ledger"

# Read back saved outputs
allocations = spark.read.parquet(ALLOCATIONS_PATH)
status = spark.read.parquet(STATUS_PATH)
ledger = spark.read.parquet(LEDGER_PATH)

cancellations = ledger.filter(F.col("transaction_type") == "CANCELLATION")
sales = ledger.filter(F.col("transaction_type") == "SALE")
cancellation_count = cancellations.count()

alloc_count = allocations.count()
status_count = status.count()

print(f"Read-back: {alloc_count:,} allocations, {status_count:,} statuses")
print(f"Source cancellations: {cancellation_count:,}")

# 1: Each cancellation has exactly one status record
assert status_count == cancellation_count, (
    f"Status count {status_count} != cancellation count {cancellation_count}"
)
distinct_status_ids = status.select("cancellation_row_id").distinct().count()
assert distinct_status_ids == status_count
print("PASS: Each cancellation has exactly one status record")

# 2: requested = matched + unmatched
bad_recon = status.filter(
    F.col("requested_quantity")
    != F.col("matched_quantity") + F.col("unmatched_quantity")
)
assert bad_recon.limit(1).count() == 0
print("PASS: requested = matched + unmatched for all cancellations")

# 3: Allocation quantities strictly positive
assert allocations.filter(
    F.col("allocated_quantity") <= 0
).limit(1).count() == 0
print("PASS: All allocation quantities strictly positive")

# 4: Sale timestamps strictly earlier
assert allocations.filter(
    F.col("sale_date") >= F.col("cancellation_date")
).limit(1).count() == 0
print("PASS: Sale timestamps strictly earlier than cancellation timestamps")

# 5: No sale over-allocated
sale_consumption = allocations.groupBy("sale_row_id").agg(
    F.sum("allocated_quantity").alias("consumed_quantity")
)
sale_quantities = sales.select(
    F.col("row_id").alias("sale_row_id"),
    F.col("quantity").cast("long").alias("sold_quantity"),
)
overalloc = sale_consumption.join(
    sale_quantities, on="sale_row_id", how="inner"
).filter(F.col("consumed_quantity") > F.col("sold_quantity"))
assert overalloc.limit(1).count() == 0
print("PASS: No sale allocated beyond original sold quantity")

# 6: Allocation totals reconcile with matched quantities
allocated_total = allocations.agg(
    F.coalesce(
        F.sum("allocated_quantity"), F.lit(0).cast("long")
    ).alias("total")
).first()["total"]
matched_total = status.agg(
    F.coalesce(
        F.sum("matched_quantity"), F.lit(0).cast("long")
    ).alias("total")
).first()["total"]
assert allocated_total == matched_total
print(f"PASS: Allocation total ({allocated_total:,}) = matched total ({matched_total:,})")

# 7: Allocation IDs unique
assert allocations.select("allocation_id").distinct().count() == alloc_count
print(f"PASS: All {alloc_count:,} allocation IDs unique")

# 8: No source cancellation missing from status
source_ids = cancellations.select("row_id")
status_ids = status.select("cancellation_row_id")
missing = source_ids.join(
    status_ids,
    source_ids.row_id == status_ids.cancellation_row_id,
    "left_anti",
)
assert missing.count() == 0
print("PASS: No source cancellation missing from status output")

# 9: No null-customer allocation
assert allocations.filter(
    F.col("customer_id").isNull()
).limit(1).count() == 0
print("PASS: No null-customer cancellation allocated")

# 10: Null-customer statuses all UNMATCHED / MISSING_CUSTOMER_ID
null_status = status.filter(
    (F.col("customer_id").isNull())
    & (F.col("cancellation_status") == "UNMATCHED")
    & (F.col("unmatched_reason") == "MISSING_CUSTOMER_ID")
)
null_count = null_status.count()
assert null_count > 0
print(f"PASS: {null_count} null-customer cancellations all UNMATCHED / MISSING_CUSTOMER_ID")

print("\n" + "=" * 60)
print("ALL 10 POST-WRITE VALIDATIONS PASSED")
print("=" * 60)

unmatched_total = status.agg(
    F.coalesce(
        F.sum("unmatched_quantity"), F.lit(0).cast("long")
    ).alias("total")
).first()["total"]

print(f"\nCancellations processed:  {status_count:,}")
print(f"Allocation records:       {alloc_count:,}")
print(f"Allocated units:           {allocated_total:,}")
print(f"Matched units:             {matched_total:,}")
print(f"Unmatched units:           {unmatched_total:,}")

In [0]:
# ============================================================
# RetailRecon | Phase 5: Financial Reporting & Customer History
# Development only. No production deployment.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType,
    LongType,
    IntegerType,
    DateType,
)

BASE_PATH = "/Volumes/portfolio_projects/retail_sales/raw"

LEDGER_PATH = f"{BASE_PATH}/classified_sales_ledger"
ADJUSTMENTS_PATH = f"{BASE_PATH}/classified_adjustments"
EXCEPTIONS_PATH = f"{BASE_PATH}/classified_exceptions"
ALLOCATIONS_PATH = f"{BASE_PATH}/cancellation_allocations_parquet"
STATUS_PATH = f"{BASE_PATH}/cancellation_status_parquet"

INVOICE_SUMMARY_PATH = f"{BASE_PATH}/invoice_summary_parquet"
MONTHLY_COUNTRY_PATH = f"{BASE_PATH}/monthly_country_sales_parquet"
CUSTOMER_HISTORY_PATH = f"{BASE_PATH}/customer_purchase_history_parquet"

# ------------------------------------------------------------
# 1. Inspect input schemas
# ------------------------------------------------------------

ledger = spark.read.parquet(LEDGER_PATH)
adjustments = spark.read.parquet(ADJUSTMENTS_PATH)
exceptions = spark.read.parquet(EXCEPTIONS_PATH)

print("=== classified_sales_ledger schema ===")
ledger.printSchema()
print(f"\nLedger row count: {ledger.count():,}")

print("\n=== Transaction type distribution ===")
ledger.groupBy("transaction_type").count().show(truncate=False)

print("\n=== Sample SALE row ===")
ledger.filter(F.col("transaction_type") == "SALE").show(
    1, truncate=False, vertical=True
)

print("\n=== Sample CANCELLATION row ===")
ledger.filter(F.col("transaction_type") == "CANCELLATION").show(
    1, truncate=False, vertical=True
)

In [0]:
# ============================================================
# RetailRecon | Phase 5: Deliverables A & B
# Invoice Summary + Monthly-Country Sales
# ============================================================

ledger = spark.read.parquet(LEDGER_PATH)
ZERO_DEC = F.lit(0).cast('decimal(30,4)')

# ------------------------------------------------------------
# Deliverable A: Invoice Summary
# One record per distinct invoice.
# Only SALE and CANCELLATION lines contribute.
# ------------------------------------------------------------

invoice_summary = (
    ledger
    .groupBy('invoice')
    .agg(
        F.min('invoice_date').alias('invoice_date'),
        F.max('country').alias('country'),
        F.sum(
            F.when(F.col('transaction_type') == 'SALE',
                   F.col('signed_line_value'))
            .otherwise(ZERO_DEC)
        ).alias('gross_sales_value'),
        F.sum(
            F.when(F.col('transaction_type') == 'CANCELLATION',
                   F.abs(F.col('signed_line_value')))
            .otherwise(ZERO_DEC)
        ).alias('cancellation_value'),
        F.sum(
            F.when(F.col('transaction_type') == 'SALE', F.lit(1))
            .otherwise(F.lit(0))
        ).alias('sales_line_count'),
        F.sum(
            F.when(F.col('transaction_type') == 'CANCELLATION', F.lit(1))
            .otherwise(F.lit(0))
        ).alias('cancellation_line_count'),
        F.sum(
            F.when(F.col('transaction_type') == 'SALE',
                   F.col('quantity').cast('long'))
            .otherwise(F.lit(0))
        ).alias('gross_sold_quantity'),
        F.sum(
            F.when(F.col('transaction_type') == 'CANCELLATION',
                   F.abs(F.col('quantity').cast('long')))
            .otherwise(F.lit(0))
        ).alias('abs_cancelled_quantity'),
    )
    .withColumn('net_sales_value',
        (F.col('gross_sales_value') - F.col('cancellation_value')).cast('decimal(30,4)'))
)

(invoice_summary.write.format('parquet').mode('overwrite')
    .option('compression', 'snappy').save(INVOICE_SUMMARY_PATH))

invoice_count = invoice_summary.count()
print(f'Deliverable A: Invoice Summary')
print(f'  Records: {invoice_count:,}')
print(f'  Path:    {INVOICE_SUMMARY_PATH}')

# ------------------------------------------------------------
# Deliverable B: Monthly-Country Sales
# Null countries replaced with UNKNOWN.
# ------------------------------------------------------------

monthly_country_sales = (
    ledger
    .withColumn('country_resolved',
        F.coalesce(F.col('country'), F.lit('UNKNOWN')))
    .groupBy('invoice_year', 'invoice_month', 'country_resolved')
    .agg(
        F.sum(
            F.when(F.col('transaction_type') == 'SALE',
                   F.col('signed_line_value'))
            .otherwise(ZERO_DEC)
        ).alias('gross_sales_value'),
        F.sum(
            F.when(F.col('transaction_type') == 'CANCELLATION',
                   F.abs(F.col('signed_line_value')))
            .otherwise(ZERO_DEC)
        ).alias('cancellation_value'),
        F.sum(
            F.when(F.col('transaction_type') == 'SALE', F.lit(1))
            .otherwise(F.lit(0))
        ).alias('sales_line_count'),
        F.sum(
            F.when(F.col('transaction_type') == 'CANCELLATION', F.lit(1))
            .otherwise(F.lit(0))
        ).alias('cancellation_line_count'),
        F.sum(
            F.when(F.col('transaction_type') == 'SALE',
                   F.col('quantity').cast('long'))
            .otherwise(F.lit(0))
        ).alias('gross_sold_quantity'),
        F.sum(
            F.when(F.col('transaction_type') == 'CANCELLATION',
                   F.abs(F.col('quantity').cast('long')))
            .otherwise(F.lit(0))
        ).alias('abs_cancelled_quantity'),
    )
    .withColumn('net_sales_value',
        (F.col('gross_sales_value') - F.col('cancellation_value')).cast('decimal(30,4)'))
    .withColumnRenamed('country_resolved', 'country')
)

(monthly_country_sales.write.format('parquet').mode('overwrite')
    .option('compression', 'snappy').save(MONTHLY_COUNTRY_PATH))

monthly_count = monthly_country_sales.count()
print(f'\nDeliverable B: Monthly-Country Sales')
print(f'  Records: {monthly_count:,}')
print(f'  Path:    {MONTHLY_COUNTRY_PATH}')

# ------------------------------------------------------------
# Cross-check financial totals against ledger
# ------------------------------------------------------------

ledger_totals = ledger.agg(
    F.sum(F.when(F.col('transaction_type') == 'SALE',
                F.col('signed_line_value')).otherwise(ZERO_DEC)).alias('gross'),
    F.sum(F.when(F.col('transaction_type') == 'CANCELLATION',
                F.abs(F.col('signed_line_value'))).otherwise(ZERO_DEC)).alias('cancel'),
).first()
net_total = ledger_totals['gross'] - ledger_totals['cancel']
print(f'\n=== Ledger Financial Totals ===')
print(f'  Gross sales:   GBP {ledger_totals["gross"]:,.2f}')
print(f'  Cancellation:  GBP {ledger_totals["cancel"]:,.2f}')
print(f'  Net sales:     GBP {net_total:,.2f}')

inv_totals = invoice_summary.agg(
    F.sum('gross_sales_value').alias('gross'),
    F.sum('cancellation_value').alias('cancel')).first()
assert inv_totals['gross'] == ledger_totals['gross']
assert inv_totals['cancel'] == ledger_totals['cancel']

mc_totals = monthly_country_sales.agg(
    F.sum('gross_sales_value').alias('gross'),
    F.sum('cancellation_value').alias('cancel')).first()
assert mc_totals['gross'] == ledger_totals['gross']
assert mc_totals['cancel'] == ledger_totals['cancel']

print('\nPASS: Invoice and monthly-country totals match ledger.')

In [0]:
# ============================================================
# RetailRecon | Phase 5: Deliverable C
# Customer Purchase History
# ============================================================
# One row per distinct identified customer purchase invoice.
# Excludes null customer IDs.
# Lifetime values are cumulative from the ledger (not FIFO allocations).
# Same-timestamp transactions are bucketed to avoid leakage.
# Deterministic tie-breaking: (purchase_invoice_timestamp, purchase_invoice).
# ============================================================

ledger = spark.read.parquet(LEDGER_PATH)

# ------------------------------------------------------------
# 1. Per-customer, per-timestamp financial aggregates
# ------------------------------------------------------------
# SALE gross value per customer per timestamp
sale_ts = (
    ledger
    .filter(F.col('transaction_type') == 'SALE')
    .filter(F.col('customer_id').isNotNull())
    .groupBy('customer_id', 'invoice_date')
    .agg(F.sum('signed_line_value').alias('gross_at_ts'))
)

# Cancellation value per customer per timestamp
# Only non-null customer cancellations (matched or unmatched)
cancel_ts = (
    ledger
    .filter(F.col('transaction_type') == 'CANCELLATION')
    .filter(F.col('customer_id').isNotNull())
    .groupBy('customer_id', 'invoice_date')
    .agg(F.sum(F.abs(F.col('signed_line_value'))).alias('cancel_at_ts'))
)

# Full outer join: all event timestamps per customer
all_ts = (
    sale_ts
    .join(cancel_ts, ['customer_id', 'invoice_date'], 'full_outer')
    .withColumn('gross_at_ts',
        F.coalesce(F.col('gross_at_ts'), F.lit(0).cast('decimal(30,4)')))
    .withColumn('cancel_at_ts',
        F.coalesce(F.col('cancel_at_ts'), F.lit(0).cast('decimal(30,4)')))
)

# ------------------------------------------------------------
# 2. Cumulative lifetime values (window function)
# ------------------------------------------------------------
# rowsBetween is safe here because all_ts has exactly one row
# per (customer_id, invoice_date) after the groupBy aggregation.

w_cum = (
    Window
    .partitionBy('customer_id')
    .orderBy('invoice_date')
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

all_ts_cum = (
    all_ts
    .withColumn('cum_gross', F.sum('gross_at_ts').over(w_cum))
    .withColumn('cum_cancel', F.sum('cancel_at_ts').over(w_cum))
)

# ------------------------------------------------------------
# 3. One row per distinct customer purchase invoice
# ------------------------------------------------------------

purchase_invoices = (
    ledger
    .filter(F.col('transaction_type') == 'SALE')
    .filter(F.col('customer_id').isNotNull())
    .groupBy('customer_id', 'invoice')
    .agg(
        F.min('invoice_date').alias('purchase_invoice_timestamp'),
        F.max('country').alias('purchase_country'),
        F.sum('signed_line_value').alias('invoice_gross_value'),
    )
)

# ------------------------------------------------------------
# 4. Join with cumulative lifetime values
# ------------------------------------------------------------
# Each purchase invoice gets the cumulative totals at its timestamp.
# Invoices sharing a timestamp get the same lifetime values
# (time-bucket approach per requirement).

customer_history = (
    purchase_invoices.alias('pi')
    .join(
        all_ts_cum.alias('ts'),
        (F.col('pi.customer_id') == F.col('ts.customer_id'))
        & (F.col('pi.purchase_invoice_timestamp') == F.col('ts.invoice_date')),
        'left'
    )
    .select(
        F.col('pi.customer_id').alias('customer_id'),
        F.col('pi.invoice').alias('purchase_invoice'),
        F.col('pi.purchase_invoice_timestamp'),
        F.col('pi.purchase_country'),
        F.col('pi.invoice_gross_value').cast('decimal(30,4)'),
        F.coalesce(
            F.col('ts.cum_gross'),
            F.col('pi.invoice_gross_value')
        ).alias('lifetime_gross_sales_value'),
        F.coalesce(
            F.col('ts.cum_cancel'),
            F.lit(0).cast('decimal(30,4)')
        ).alias('lifetime_cancellation_value'),
    )
)

# ------------------------------------------------------------
# 5. Purchase sequence fields (window functions)
# ------------------------------------------------------------
# Deterministic ordering: (purchase_invoice_timestamp, purchase_invoice)

w_order = (
    Window
    .partitionBy('customer_id')
    .orderBy('purchase_invoice_timestamp', 'purchase_invoice')
)

customer_history = (
    customer_history
    .withColumn(
        'first_purchase_date',
        F.min('purchase_invoice_timestamp').over(
            Window.partitionBy('customer_id')
        )
    )
    .withColumn(
        'previous_purchase_invoice_date',
        F.lag('purchase_invoice_timestamp').over(w_order)
    )
    .withColumn(
        'days_between_purchases',
        F.when(
            F.col('previous_purchase_invoice_date').isNotNull(),
            F.datediff(
                'purchase_invoice_timestamp',
                'previous_purchase_invoice_date'
            )
        ).otherwise(F.lit(None).cast('int'))
    )
)

# Final column order
customer_history = customer_history.select(
    'customer_id',
    'purchase_invoice',
    'purchase_invoice_timestamp',
    'purchase_country',
    'invoice_gross_value',
    'first_purchase_date',
    'previous_purchase_invoice_date',
    'days_between_purchases',
    'lifetime_gross_sales_value',
    'lifetime_cancellation_value',
)

# ------------------------------------------------------------
# 6. Write customer purchase history
# ------------------------------------------------------------

(customer_history.write.format('parquet').mode('overwrite')
    .option('compression', 'snappy').save(CUSTOMER_HISTORY_PATH))

history_count = customer_history.count()
customer_count = customer_history.select('customer_id').distinct().count()

print(f'Deliverable C: Customer Purchase History')
print(f'  Records:           {history_count:,}')
print(f'  Distinct customers: {customer_count:,}')
print(f'  Path:              {CUSTOMER_HISTORY_PATH}')

# Show sample
print(f'\nSample (first 5 rows ordered by customer_id, timestamp):')
customer_history.orderBy('customer_id', 'purchase_invoice_timestamp', 'purchase_invoice').show(5, truncate=False)

In [0]:
# ============================================================
# RetailRecon | Phase 5: Financial Reconciliation Tests A-F
# Reads all three persisted outputs from disk and validates.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE_PATH = "/Volumes/portfolio_projects/retail_sales/raw"

LEDGER_PATH = f"{BASE_PATH}/classified_sales_ledger"
INVOICE_SUMMARY_PATH = f"{BASE_PATH}/invoice_summary_parquet"
MONTHLY_COUNTRY_PATH = f"{BASE_PATH}/monthly_country_sales_parquet"
CUSTOMER_HISTORY_PATH = f"{BASE_PATH}/customer_purchase_history_parquet"

ZERO_DEC = F.lit(0).cast("decimal(30,4)")

# Read back ALL outputs from disk (Test F requirement)
ledger = spark.read.parquet(LEDGER_PATH)
invoice_summary = spark.read.parquet(INVOICE_SUMMARY_PATH)
monthly_country = spark.read.parquet(MONTHLY_COUNTRY_PATH)
customer_history = spark.read.parquet(CUSTOMER_HISTORY_PATH)

inv_count = invoice_summary.count()
mc_count = monthly_country.count()
ch_count = customer_history.count()

print(f"Read back: {inv_count:,} invoices, {mc_count:,} monthly-country, {ch_count:,} customer-history")

# ------------------------------------------------------------
# Test A: Invoice to monthly-country reconciliation
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("Test A: Invoice to monthly-country reconciliation")
print("=" * 60)

inv_to_monthly = (
    invoice_summary
    .withColumn("invoice_year", F.year("invoice_date"))
    .withColumn("invoice_month", F.month("invoice_date"))
    .withColumn(
        "country_resolved",
        F.coalesce(F.col("country"), F.lit("UNKNOWN")),
    )
    .groupBy("invoice_year", "invoice_month", "country_resolved")
    .agg(
        F.sum("gross_sales_value").alias("inv_gross"),
        F.sum("cancellation_value").alias("inv_cancel"),
        F.sum("net_sales_value").alias("inv_net"),
    )
)

mc = monthly_country.select(
    F.col("invoice_year"),
    F.col("invoice_month"),
    F.col("country").alias("country_resolved"),
    F.col("gross_sales_value").alias("mc_gross"),
    F.col("cancellation_value").alias("mc_cancel"),
    F.col("net_sales_value").alias("mc_net"),
)

diff_a = (
    inv_to_monthly
    .join(mc, ["invoice_year", "invoice_month", "country_resolved"], "full_outer")
    .filter(
        (
            F.coalesce(F.col("inv_gross"), ZERO_DEC)
            != F.coalesce(F.col("mc_gross"), ZERO_DEC)
        )
        | (
            F.coalesce(F.col("inv_cancel"), ZERO_DEC)
            != F.coalesce(F.col("mc_cancel"), ZERO_DEC)
        )
        | (
            F.coalesce(F.col("inv_net"), ZERO_DEC)
            != F.coalesce(F.col("mc_net"), ZERO_DEC)
        )
    )
)
diff_a_count = diff_a.count()
assert diff_a_count == 0, f"Test A failed: {diff_a_count} groups differ"
print("PASS: Invoice-to-monthly reconciliation matches for all groups")

# ------------------------------------------------------------
# Test B: Net-value identity
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("Test B: Net-value identity")
print("=" * 60)

bad_inv = invoice_summary.filter(
    F.col("net_sales_value")
    != (F.col("gross_sales_value") - F.col("cancellation_value"))
)
assert bad_inv.limit(1).count() == 0, "Net value identity failed for invoice summary"
print("PASS: net = gross - cancellation for all invoices")

bad_mc = monthly_country.filter(
    F.col("net_sales_value")
    != (F.col("gross_sales_value") - F.col("cancellation_value"))
)
assert bad_mc.limit(1).count() == 0, "Net value identity failed for monthly-country"
print("PASS: net = gross - cancellation for all monthly-country groups")

# ------------------------------------------------------------
# Test C: Source row conservation
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("Test C: Source row conservation")
print("=" * 60)

ledger_sales_count = ledger.filter(
    F.col("transaction_type") == "SALE"
).count()
ledger_cancel_count = ledger.filter(
    F.col("transaction_type") == "CANCELLATION"
).count()

inv_sales_count = invoice_summary.agg(
    F.sum("sales_line_count")
).first()[0]
inv_cancel_count = invoice_summary.agg(
    F.sum("cancellation_line_count")
).first()[0]

mc_sales_count = monthly_country.agg(
    F.sum("sales_line_count")
).first()[0]
mc_cancel_count = monthly_country.agg(
    F.sum("cancellation_line_count")
).first()[0]

assert inv_sales_count == ledger_sales_count, (
    f"Invoice sales count {inv_sales_count:,} != ledger {ledger_sales_count:,}"
)
assert inv_cancel_count == ledger_cancel_count, (
    f"Invoice cancel count {inv_cancel_count:,} != ledger {ledger_cancel_count:,}"
)
assert mc_sales_count == ledger_sales_count, (
    f"Monthly sales count {mc_sales_count:,} != ledger {ledger_sales_count:,}"
)
assert mc_cancel_count == ledger_cancel_count, (
    f"Monthly cancel count {mc_cancel_count:,} != ledger {ledger_cancel_count:,}"
)
print(
    f"PASS: Line counts reconcile "
    f"({ledger_sales_count:,} sales, {ledger_cancel_count:,} cancellations)"
)

# ------------------------------------------------------------
# Test D: Cancellation inclusion
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("Test D: Cancellation inclusion")
print("=" * 60)

ledger_cancel_value = ledger.filter(
    F.col("transaction_type") == "CANCELLATION"
).agg(F.sum(F.abs(F.col("signed_line_value")))).first()[0]

ledger_cancel_qty = ledger.filter(
    F.col("transaction_type") == "CANCELLATION"
).agg(F.sum(F.abs(F.col("quantity").cast("long")))).first()[0]

inv_cancel_value = invoice_summary.agg(
    F.sum("cancellation_value")
).first()[0]
inv_cancel_qty = invoice_summary.agg(
    F.sum("abs_cancelled_quantity")
).first()[0]

mc_cancel_value = monthly_country.agg(
    F.sum("cancellation_value")
).first()[0]
mc_cancel_qty = monthly_country.agg(
    F.sum("abs_cancelled_quantity")
).first()[0]

assert inv_cancel_value == ledger_cancel_value, (
    f"Invoice cancel value {inv_cancel_value} != ledger {ledger_cancel_value}"
)
assert inv_cancel_qty == ledger_cancel_qty, (
    f"Invoice cancel qty {inv_cancel_qty} != ledger {ledger_cancel_qty}"
)
assert mc_cancel_value == ledger_cancel_value, (
    f"Monthly cancel value {mc_cancel_value} != ledger {ledger_cancel_value}"
)
assert mc_cancel_qty == ledger_cancel_qty, (
    f"Monthly cancel qty {mc_cancel_qty} != ledger {ledger_cancel_qty}"
)
print(
    f"PASS: Cancellation value (GBP {ledger_cancel_value:,.2f}) "
    f"and quantity ({ledger_cancel_qty:,}) reconcile"
)

# ------------------------------------------------------------
# Test E: Customer history validation
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("Test E: Customer history validation")
print("=" * 60)

# One row per distinct customer purchase invoice
dup_count = (
    customer_history
    .groupBy("customer_id", "purchase_invoice")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
assert dup_count == 0, f"Duplicate customer/invoice keys: {dup_count}"

# No null customer IDs
null_customers = customer_history.filter(
    F.col("customer_id").isNull()
).count()
assert null_customers == 0, f"Null customer IDs in history: {null_customers}"

# First purchase date is the earliest for each customer
bad_first = (
    customer_history
    .withColumn(
        "computed_first",
        F.min("purchase_invoice_timestamp").over(
            Window.partitionBy("customer_id")
        ),
    )
    .filter(F.col("first_purchase_date") != F.col("computed_first"))
)
assert bad_first.limit(1).count() == 0, "First purchase date mismatch"

# Previous purchase date from preceding invoice
w_order = (
    Window
    .partitionBy("customer_id")
    .orderBy("purchase_invoice_timestamp", "purchase_invoice")
)
bad_prev = (
    customer_history
    .withColumn(
        "expected_prev",
        F.lag("purchase_invoice_timestamp").over(w_order),
    )
    .filter(
        F.coalesce(
            F.col("previous_purchase_invoice_date"),
            F.lit(None).cast("timestamp"),
        )
        != F.coalesce(
            F.col("expected_prev"),
            F.lit(None).cast("timestamp"),
        )
    )
)
assert bad_prev.limit(1).count() == 0, "Previous purchase date mismatch"

# Days between purchases
bad_days = customer_history.filter(
    F.col("previous_purchase_invoice_date").isNotNull()
    & (
        F.col("days_between_purchases")
        != F.datediff(
            "purchase_invoice_timestamp",
            "previous_purchase_invoice_date",
        )
    )
)
assert bad_days.limit(1).count() == 0, "Days between purchases mismatch"

# Lifetime values are non-negative
bad_neg_gross = customer_history.filter(
    F.col("lifetime_gross_sales_value") < 0
)
assert bad_neg_gross.limit(1).count() == 0, "Negative lifetime gross found"

bad_neg_cancel = customer_history.filter(
    F.col("lifetime_cancellation_value") < 0
)
assert bad_neg_cancel.limit(1).count() == 0, "Negative lifetime cancel found"

# Lifetime values are monotonically non-decreasing per customer
w_seq = (
    Window
    .partitionBy("customer_id")
    .orderBy("purchase_invoice_timestamp", "purchase_invoice")
)
bad_monotone = (
    customer_history
    .withColumn(
        "prev_lifetime_gross",
        F.lag("lifetime_gross_sales_value").over(w_seq),
    )
    .filter(
        F.col("prev_lifetime_gross").isNotNull()
        & (
            F.col("lifetime_gross_sales_value")
            < F.col("prev_lifetime_gross")
        )
    )
)
assert bad_monotone.limit(1).count() == 0, (
    "Lifetime gross is not monotonically non-decreasing"
)

# Final lifetime gross per customer must not exceed total gross sales
# (no future leakage — cumulative stops at last purchase timestamp).
# Note: invoices with lines at multiple timestamps may cause the last
# purchase's cumulative to be < total, because later lines from earlier
# invoices fall after the last purchase's timestamp. This is correct:
# lifetime is cumulative AS OF the purchase timestamp.
total_gross_per_customer = (
    ledger
    .filter(F.col("transaction_type") == "SALE")
    .filter(F.col("customer_id").isNotNull())
    .groupBy("customer_id")
    .agg(F.sum("signed_line_value").alias("total_gross"))
)

# No row should have lifetime_gross > total_gross (would mean future leakage)
leak_check = (
    customer_history
    .join(total_gross_per_customer, "customer_id", "inner")
    .filter(F.col("lifetime_gross_sales_value") > F.col("total_gross"))
)
assert leak_check.limit(1).count() == 0, (
    "Lifetime gross exceeds total gross sales — future data leaked"
)

# Exactly one row per customer has null previous date (the first purchase)
null_prev_count = (
    customer_history
    .filter(F.col("previous_purchase_invoice_date").isNull())
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)
assert null_prev_count.limit(1).count() == 0, (
    "Multiple rows with null previous date for same customer"
)

# The row with null previous date must be the first purchase
bad_first = (
    customer_history
    .filter(F.col("previous_purchase_invoice_date").isNull())
    .filter(F.col("purchase_invoice_timestamp") != F.col("first_purchase_date"))
)
assert bad_first.limit(1).count() == 0, (
    "Row with null previous date is not the first purchase"
)

# The first purchase also has null days_between_purchases
bad_first_days = (
    customer_history
    .filter(F.col("previous_purchase_invoice_date").isNull())
    .filter(F.col("days_between_purchases").isNotNull())
)
assert bad_first_days.limit(1).count() == 0, (
    "First purchase should have null days_between_purchases"
)

print(
    "PASS: Customer history validated "
    "(unique keys, first/prev dates, intervals, "
    "non-decreasing lifetime values, final-value reconciliation)"
)

# ------------------------------------------------------------
# Test F: Persisted-output checks
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("Test F: Persisted-output checks")
print("=" * 60)

# All three outputs non-empty
assert inv_count > 0, "Invoice summary is empty"
assert mc_count > 0, "Monthly-country sales is empty"
assert ch_count > 0, "Customer purchase history is empty"

# Unique invoice keys
inv_distinct = invoice_summary.select("invoice").distinct().count()
assert inv_count == inv_distinct, (
    f"Duplicate invoices: {inv_count} vs {inv_distinct}"
)

# Unique customer/invoice keys
ch_distinct = (
    customer_history
    .select("customer_id", "purchase_invoice")
    .distinct()
    .count()
)
assert ch_count == ch_distinct, (
    f"Duplicate customer/invoice keys: {ch_count} vs {ch_distinct}"
)

# Schema type checks: monetary values must be decimal, not float
for col_name in ["gross_sales_value", "cancellation_value", "net_sales_value"]:
    field = [f for f in invoice_summary.schema.fields if f.name == col_name][0]
    assert str(field.dataType).lower().startswith("decimal"), (
        f"invoice_summary.{col_name} is {field.dataType}, expected decimal"
    )

for col_name in ["gross_sales_value", "cancellation_value", "net_sales_value"]:
    field = [f for f in monthly_country.schema.fields if f.name == col_name][0]
    assert str(field.dataType).lower().startswith("decimal"), (
        f"monthly_country.{col_name} is {field.dataType}, expected decimal"
    )

for col_name in [
    "invoice_gross_value",
    "lifetime_gross_sales_value",
    "lifetime_cancellation_value",
]:
    field = [f for f in customer_history.schema.fields if f.name == col_name][0]
    assert str(field.dataType).lower().startswith("decimal"), (
        f"customer_history.{col_name} is {field.dataType}, expected decimal"
    )

print("PASS: All persisted outputs read back with correct schemas and unique keys")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("ALL 6 FINANCIAL RECONCILIATION TESTS PASSED")
print("=" * 60)

customer_count = customer_history.select("customer_id").distinct().count()
purchase_invoice_count = customer_history.select(
    "purchase_invoice"
).distinct().count()

gross_total = invoice_summary.agg(F.sum("gross_sales_value")).first()[0]
cancel_total = invoice_summary.agg(F.sum("cancellation_value")).first()[0]
net_total = gross_total - cancel_total

print(f"\nInvoice summary records:     {inv_count:,}")
print(f"Monthly-country records:     {mc_count:,}")
print(f"Customer purchase history:   {ch_count:,}")
print(f"Distinct customers:          {customer_count:,}")
print(f"Distinct purchase invoices:  {purchase_invoice_count:,}")
print(f"\nGross sales value:   GBP {gross_total:,.2f}")
print(f"Cancellation value:  GBP {cancel_total:,.2f}")
print(f"Net sales value:     GBP {net_total:,.2f}")


In [0]:
# ============================================================
# RetailRecon | Phase 6: Deliverable A
# End-to-End Row-Accounting & Quality Gates
# ============================================================
# 14 executable validation checks spanning the entire pipeline.
# All counts and totals are read from persisted Parquet datasets.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE_PATH = "/Volumes/portfolio_projects/retail_sales/raw"

RAW_PATH = f"{BASE_PATH}/source_raw_parquet"
TYPED_PATH = f"{BASE_PATH}/source_typed_parquet"
EXCLUSIONS_PATH = f"{BASE_PATH}/source_overlap_exclusions_parquet"
LEDGER_PATH = f"{BASE_PATH}/classified_sales_ledger"
ADJUSTMENTS_PATH = f"{BASE_PATH}/classified_adjustments"
EXCEPTIONS_PATH = f"{BASE_PATH}/classified_exceptions"
ALLOCATIONS_PATH = f"{BASE_PATH}/cancellation_allocations_parquet"
STATUS_PATH = f"{BASE_PATH}/cancellation_status_parquet"
INVOICE_SUMMARY_PATH = f"{BASE_PATH}/invoice_summary_parquet"
MONTHLY_COUNTRY_PATH = f"{BASE_PATH}/monthly_country_sales_parquet"
CUSTOMER_HISTORY_PATH = f"{BASE_PATH}/customer_purchase_history_parquet"

ZERO_DEC = F.lit(0).cast("decimal(30,4)")

# Read all datasets
raw = spark.read.parquet(RAW_PATH)
typed = spark.read.parquet(TYPED_PATH)
exclusions = spark.read.parquet(EXCLUSIONS_PATH)
ledger = spark.read.parquet(LEDGER_PATH)
adjustments = spark.read.parquet(ADJUSTMENTS_PATH)
exceptions = spark.read.parquet(EXCEPTIONS_PATH)
allocations = spark.read.parquet(ALLOCATIONS_PATH)
status = spark.read.parquet(STATUS_PATH)
invoice_summary = spark.read.parquet(INVOICE_SUMMARY_PATH)
monthly_country = spark.read.parquet(MONTHLY_COUNTRY_PATH)
customer_history = spark.read.parquet(CUSTOMER_HISTORY_PATH)

# Count all datasets
raw_count = raw.count()
typed_count = typed.count()
exclusion_count = exclusions.count()
ledger_count = ledger.count()
adjustment_count = adjustments.count()
exception_count = exceptions.count()
sales_count = ledger.filter(F.col("transaction_type") == "SALE").count()
cancel_count = ledger.filter(F.col("transaction_type") == "CANCELLATION").count()
alloc_count = allocations.count()
status_count = status.count()
inv_summ_count = invoice_summary.count()
mc_count = monthly_country.count()
ch_count = customer_history.count()

gross_total = invoice_summary.agg(F.sum("gross_sales_value")).first()[0]
cancel_total = invoice_summary.agg(F.sum("cancellation_value")).first()[0]
net_total = gross_total - cancel_total

print("=" * 70)
print("PHASE 6 DELIVERABLE A: End-to-End Row-Accounting Checks")
print("=" * 70)
print(f"\nBaseline counts:")
print(f"  Raw source rows:          {raw_count:,}")
print(f"  Typed retained rows:      {typed_count:,}")
print(f"  Overlap exclusions:       {exclusion_count:,}")
print(f"  Merchandise ledger:       {ledger_count:,}")
print(f"  Adjustments:              {adjustment_count:,}")
print(f"  Exceptions:               {exception_count:,}")
print(f"  Sale rows:                {sales_count:,}")
print(f"  Cancellation rows:         {cancel_count:,}")
print(f"  FIFO allocations:         {alloc_count:,}")
print(f"  Cancellation statuses:    {status_count:,}")
print(f"  Invoice summaries:        {inv_summ_count:,}")
print(f"  Monthly-country summaries: {mc_count:,}")
print(f"  Customer history rows:    {ch_count:,}")
print(f"\nFinancial totals:")
print(f"  Gross sales:   GBP {gross_total:,.2f}")
print(f"  Cancellation:  GBP {cancel_total:,.2f}")
print(f"  Net sales:     GBP {net_total:,.2f}")

checks_passed = 0
checks_total = 14

# ------------------------------------------------------------
# Check 1: Raw rows reconcile to typed + exclusions
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 1: Raw rows = typed retained + overlap exclusions")
assert raw_count == typed_count + exclusion_count, (
    f"FAIL: {raw_count:,} != {typed_count:,} + {exclusion_count:,} "
    f"= {typed_count + exclusion_count:,}"
)
print(f"PASS: {raw_count:,} = {typed_count:,} + {exclusion_count:,}")
checks_passed += 1

# ------------------------------------------------------------
# Check 2: Typed rows reconcile to ledger + adjustments + exceptions
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 2: Typed rows = ledger + adjustments + exceptions")
assert typed_count == ledger_count + adjustment_count + exception_count, (
    f"FAIL: {typed_count:,} != {ledger_count:,} + {adjustment_count:,} "
    f"+ {exception_count:,} = {ledger_count + adjustment_count + exception_count:,}"
)
print(
    f"PASS: {typed_count:,} = {ledger_count:,} + {adjustment_count:,} "
    f"+ {exception_count:,}"
)
checks_passed += 1

# ------------------------------------------------------------
# Check 3: Sales + cancellations = ledger total
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 3: Sales + cancellations = merchandise ledger total")
assert sales_count + cancel_count == ledger_count, (
    f"FAIL: {sales_count:,} + {cancel_count:,} = {sales_count + cancel_count:,} "
    f"!= {ledger_count:,}"
)
print(f"PASS: {sales_count:,} + {cancel_count:,} = {ledger_count:,}")
checks_passed += 1

# ------------------------------------------------------------
# Check 4: Each source row belongs to exactly one classification output
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 4: Row ID exclusivity across classification outputs")
all_output_ids = (
    ledger.select("row_id")
    .unionByName(adjustments.select("row_id"))
    .unionByName(exceptions.select("row_id"))
)
total_output_ids = all_output_ids.count()
unique_output_ids = all_output_ids.distinct().count()
assert total_output_ids == unique_output_ids == typed_count, (
    f"FAIL: total={total_output_ids:,}, unique={unique_output_ids:,}, "
    f"typed={typed_count:,}"
)
print(
    f"PASS: {unique_output_ids:,} unique row IDs across all outputs "
    f"= typed source {typed_count:,}"
)
checks_passed += 1

# ------------------------------------------------------------
# Check 5: No duplicate or null row IDs where required
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 5: No duplicate or null row IDs")
for name, df in [
    ("ledger", ledger),
    ("adjustments", adjustments),
    ("exceptions", exceptions),
    ("allocations", allocations),
    ("status", status),
]:
    null_ids = df.filter(F.col("row_id").isNull()).limit(1).count() if "row_id" in df.columns else 0
    if "row_id" in df.columns:
        dup_ids = df.count() - df.select("row_id").distinct().count()
        assert null_ids == 0, f"FAIL: {name} has null row_id"
        assert dup_ids == 0, f"FAIL: {name} has {dup_ids} duplicate row_ids"
print("PASS: No null or duplicate row IDs in any dataset")
checks_passed += 1

# ------------------------------------------------------------
# Check 6: Every cancellation has exactly one status record
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 6: Every cancellation has exactly one status record")
assert status_count == cancel_count, (
    f"FAIL: status {status_count:,} != cancellations {cancel_count:,}"
)
distinct_status_ids = status.select("cancellation_row_id").distinct().count()
assert distinct_status_ids == status_count, (
    f"FAIL: {distinct_status_ids:,} distinct vs {status_count:,} total statuses"
)
print(f"PASS: {status_count:,} statuses = {cancel_count:,} cancellations")
checks_passed += 1

# ------------------------------------------------------------
# Check 7: requested = matched + unmatched for all cancellations
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 7: requested_quantity = matched + unmatched")
bad_recon = status.filter(
    F.col("requested_quantity")
    != F.col("matched_quantity") + F.col("unmatched_quantity")
)
assert bad_recon.limit(1).count() == 0, "FAIL: quantity reconciliation failed"
print("PASS: All cancellation quantities reconcile")
checks_passed += 1

# ------------------------------------------------------------
# Check 8: No allocation exceeds sale's remaining original quantity
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 8: No allocation exceeds sale's original quantity")
sale_consumption = (
    allocations.groupBy("sale_row_id")
    .agg(F.sum("allocated_quantity").alias("consumed"))
)
sale_quantities = (
    ledger.filter(F.col("transaction_type") == "SALE")
    .select(
        F.col("row_id").alias("sale_row_id"),
        F.col("quantity").cast("long").alias("sold"),
    )
)
overalloc = (
    sale_consumption.join(sale_quantities, "sale_row_id", "inner")
    .filter(F.col("consumed") > F.col("sold"))
)
assert overalloc.limit(1).count() == 0, "FAIL: sale over-allocated"
print("PASS: No sale allocated beyond original quantity")
checks_passed += 1

# ------------------------------------------------------------
# Check 9: Sale timestamps strictly earlier than cancellation timestamps
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 9: Sale timestamps strictly earlier than cancellation timestamps")
bad_ts = allocations.filter(F.col("sale_date") >= F.col("cancellation_date"))
assert bad_ts.limit(1).count() == 0, "FAIL: sale_date >= cancellation_date found"
print("PASS: All allocation sale timestamps strictly earlier")
checks_passed += 1

# ------------------------------------------------------------
# Check 10: Matched allocation totals = matched cancellation quantities
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 10: Allocation totals reconcile with matched quantities")
allocated_total = allocations.agg(
    F.coalesce(F.sum("allocated_quantity"), F.lit(0).cast("long")).alias("t")
).first()["t"]
matched_total = status.agg(
    F.coalesce(F.sum("matched_quantity"), F.lit(0).cast("long")).alias("t")
).first()["t"]
assert allocated_total == matched_total, (
    f"FAIL: allocated {allocated_total:,} != matched {matched_total:,}"
)
print(f"PASS: Allocated {allocated_total:,} = matched {matched_total:,}")
checks_passed += 1

# ------------------------------------------------------------
# Check 11: Missing-customer cancellations remain unmatched
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 11: Missing-customer cancellations remain unmatched")
null_status_matched = status.filter(
    (F.col("customer_id").isNull())
    & (F.col("matched_quantity") > 0)
)
assert null_status_matched.limit(1).count() == 0, (
    "FAIL: null-customer cancellation has matched > 0"
)
null_alloc = allocations.filter(F.col("customer_id").isNull())
assert null_alloc.limit(1).count() == 0, "FAIL: null-customer allocation found"
null_unmatched = status.filter(
    (F.col("customer_id").isNull())
    & (F.col("unmatched_reason") == "MISSING_CUSTOMER_ID")
).count()
assert null_unmatched > 0, "FAIL: no MISSING_CUSTOMER_ID records found"
print(f"PASS: {null_unmatched} null-customer cancellations all unmatched")
checks_passed += 1

# ------------------------------------------------------------
# Check 12: Invoice & monthly-country summaries reconcile with ledger
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 12: Invoice and monthly-country summaries reconcile with ledger")
ledger_gross = ledger.filter(F.col("transaction_type") == "SALE").agg(
    F.sum("signed_line_value")
).first()[0]
ledger_cancel_val = ledger.filter(
    F.col("transaction_type") == "CANCELLATION"
).agg(F.sum(F.abs(F.col("signed_line_value")))).first()[0]

inv_gross = invoice_summary.agg(F.sum("gross_sales_value")).first()[0]
inv_cancel = invoice_summary.agg(F.sum("cancellation_value")).first()[0]
mc_gross = monthly_country.agg(F.sum("gross_sales_value")).first()[0]
mc_cancel = monthly_country.agg(F.sum("cancellation_value")).first()[0]

assert inv_gross == ledger_gross, (
    f"FAIL: invoice gross {inv_gross} != ledger {ledger_gross}"
)
assert inv_cancel == ledger_cancel_val, (
    f"FAIL: invoice cancel {inv_cancel} != ledger {ledger_cancel_val}"
)
assert mc_gross == ledger_gross, (
    f"FAIL: monthly gross {mc_gross} != ledger {ledger_gross}"
)
assert mc_cancel == ledger_cancel_val, (
    f"FAIL: monthly cancel {mc_cancel} != ledger {ledger_cancel_val}"
)
print(
    f"PASS: Invoice & monthly-country totals match ledger "
    f"(gross={ledger_gross:,.2f}, cancel={ledger_cancel_val:,.2f})"
)
checks_passed += 1

# ------------------------------------------------------------
# Check 13: Customer purchase-history keys are unique
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 13: Customer purchase-history keys are unique")
ch_dupes = (
    customer_history
    .groupBy("customer_id", "purchase_invoice")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
assert ch_dupes == 0, f"FAIL: {ch_dupes} duplicate customer/invoice keys"
ch_distinct = (
    customer_history
    .select("customer_id", "purchase_invoice")
    .distinct()
    .count()
)
assert ch_distinct == ch_count, (
    f"FAIL: {ch_distinct:,} distinct vs {ch_count:,} total"
)
print(f"PASS: {ch_count:,} unique customer/invoice keys")
checks_passed += 1

# ------------------------------------------------------------
# Check 14: Net financial value = gross - cancellation
# ------------------------------------------------------------
print("\n" + "-" * 70)
print("Check 14: Net value = gross - cancellation")
bad_inv_net = invoice_summary.filter(
    F.col("net_sales_value")
    != F.col("gross_sales_value") - F.col("cancellation_value")
)
assert bad_inv_net.limit(1).count() == 0, "FAIL: invoice net identity"
bad_mc_net = monthly_country.filter(
    F.col("net_sales_value")
    != F.col("gross_sales_value") - F.col("cancellation_value")
)
assert bad_mc_net.limit(1).count() == 0, "FAIL: monthly-country net identity"
assert net_total == gross_total - cancel_total, (
    f"FAIL: {net_total:,.2f} != {gross_total:,.2f} - {cancel_total:,.2f}"
)
print(f"PASS: Net {net_total:,.2f} = Gross {gross_total:,.2f} - Cancel {cancel_total:,.2f}")
checks_passed += 1

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("\n" + "=" * 70)
print(f"DELIVERABLE A SUMMARY: {checks_passed}/{checks_total} checks passed")
print("=" * 70)

# Verify against recorded baselines
print("\nBaseline verification:")
baselines = {
    "Raw source rows": (raw_count, 1_067_371),
    "Typed retained rows": (typed_count, 1_044_848),
    "Merchandise ledger": (ledger_count, 1_033_442),
    "Sale rows": (sales_count, 1_015_251),
    "Cancellation rows": (cancel_count, 18_191),
    "Adjustments": (adjustment_count, 5_377),
    "Exceptions": (exception_count, 6_029),
    "FIFO allocations": (alloc_count, 16_339),
    "Cancellation statuses": (status_count, 18_191),
    "Invoice summaries": (inv_summ_count, 47_155),
    "Monthly-country": (mc_count, 585),
    "Customer history": (ch_count, 36_669),
}
all_match = True
for name, (actual, expected) in baselines.items():
    match = "OK" if actual == expected else "MISMATCH"
    if actual != expected:
        all_match = False
    print(f"  {name}: {actual:,} (expected {expected:,}) [{match}]")

if all_match:
    print("\nALL BASELINE COUNTS MATCH RECORDED VALUES")
else:
    print("\nWARNING: Some baseline counts differ from recorded values")


In [0]:
# ============================================================
# RetailRecon | Phase 6: Deliverable B
# Rerun & Idempotency Test
# ============================================================
# Re-runs FIFO allocation and financial reporting against
# unchanged inputs. Compares deterministic fingerprints.
# Uses dedicated test output paths — original outputs untouched.
# ============================================================

from collections import deque
from decimal import Decimal
from hashlib import sha256
import time

import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DecimalType, LongType, IntegerType,
)

BASE_PATH = "/Volumes/portfolio_projects/retail_sales/raw"
LEDGER_PATH = f"{BASE_PATH}/classified_sales_ledger"
TEST_STAGING = f"{BASE_PATH}/_idempotency_staging"
TEST_ALLOC = f"{BASE_PATH}/_idempotency_test_alloc"
TEST_STATUS = f"{BASE_PATH}/_idempotency_test_status"
TEST_INV_SUMM = f"{BASE_PATH}/_idempotency_test_inv_summ"
TEST_MC = f"{BASE_PATH}/_idempotency_test_mc"

ORIG_ALLOC = f"{BASE_PATH}/cancellation_allocations_parquet"
ORIG_STATUS = f"{BASE_PATH}/cancellation_status_parquet"
ORIG_INV_SUMM = f"{BASE_PATH}/invoice_summary_parquet"
ORIG_MC = f"{BASE_PATH}/monthly_country_sales_parquet"

ZERO_DEC = F.lit(0).cast("decimal(30,4)")

# ------------------------------------------------------------
# Self-contained FIFO function (identical to cell 10)
# ------------------------------------------------------------

_fifo_schema = StructType([
    StructField("record_type", StringType(), True),
    StructField("allocation_id", StringType(), True),
    StructField("cancellation_row_id", StringType(), True),
    StructField("sale_row_id", StringType(), True),
    StructField("cancellation_invoice", StringType(), True),
    StructField("sale_invoice", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("sale_stock_code", StringType(), True),
    StructField("cancellation_stock_code", StringType(), True),
    StructField("unit_price", DecimalType(18, 4), True),
    StructField("sale_date", TimestampType(), True),
    StructField("cancellation_date", TimestampType(), True),
    StructField("requested_quantity", LongType(), True),
    StructField("matched_quantity", LongType(), True),
    StructField("unmatched_quantity", LongType(), True),
    StructField("allocated_quantity", LongType(), True),
    StructField("allocation_value", DecimalType(30, 4), True),
    StructField("days_to_cancellation", IntegerType(), True),
    StructField("cancellation_status", StringType(), True),
    StructField("unmatched_reason", StringType(), True),
    StructField("eligible_prior_sales_quantity", LongType(), True),
])

_fifo_cols = [f.name for f in _fifo_schema.fields]
_fifo_int_cols = [
    "requested_quantity", "matched_quantity", "unmatched_quantity",
    "allocated_quantity", "days_to_cancellation",
    "eligible_prior_sales_quantity",
]
_fifo_ts_cols = ["sale_date", "cancellation_date"]


def _fifo_group(pdf):
    if pdf.empty:
        return pd.DataFrame(columns=_fifo_cols)
    pdf = pdf.copy()
    pdf["_event_priority"] = pdf["transaction_type"].map(
        {"CANCELLATION": 0, "SALE": 1}
    )
    pdf = pdf.sort_values(
        by=["invoice_date", "_event_priority", "invoice", "row_id"],
        ascending=True, kind="mergesort",
    )
    inventory = deque()
    records = []
    prior_sales_quantity = 0
    for event in pdf.to_dict("records"):
        event_type = event["transaction_type"]
        if event_type == "SALE":
            quantity = int(event["quantity"])
            if quantity <= 0:
                raise ValueError(f"Invalid sale quantity for row {event['row_id']}")
            inventory.append({
                "row_id": event["row_id"], "invoice": event["invoice"],
                "invoice_date": event["invoice_date"],
                "stock_code": event["stock_code"],
                "price": event["price"],
                "remaining_quantity": quantity,
            })
            prior_sales_quantity += quantity
            continue
        requested = abs(int(event["quantity"]))
        if requested <= 0:
            raise ValueError(f"Invalid cancellation quantity for row {event['row_id']}")
        remaining_request = requested
        matched = 0
        while remaining_request > 0 and inventory:
            while inventory and inventory[0]["remaining_quantity"] == 0:
                inventory.popleft()
            if not inventory:
                break
            lot = inventory[0]
            allocated = min(remaining_request, lot["remaining_quantity"])
            if allocated <= 0:
                raise ValueError("Non-positive FIFO allocation detected.")
            price = Decimal(str(event["price"]))
            allocation_value = (price * Decimal(allocated)).quantize(Decimal("0.0001"))
            sale_date = pd.Timestamp(lot["invoice_date"])
            cancellation_date = pd.Timestamp(event["invoice_date"])
            if sale_date >= cancellation_date:
                raise ValueError("FIFO attempted to match a sale at or after cancellation")
            pair_key = f"{lot['row_id']}||{event['row_id']}"
            records.append({
                "record_type": "ALLOCATION",
                "allocation_id": sha256(pair_key.encode("utf-8")).hexdigest(),
                "cancellation_row_id": event["row_id"],
                "sale_row_id": lot["row_id"],
                "cancellation_invoice": event["invoice"],
                "sale_invoice": lot["invoice"],
                "customer_id": event["customer_id"],
                "sale_stock_code": lot["stock_code"],
                "cancellation_stock_code": event["stock_code"],
                "unit_price": price,
                "sale_date": sale_date.to_pydatetime(),
                "cancellation_date": cancellation_date.to_pydatetime(),
                "requested_quantity": None, "matched_quantity": None,
                "unmatched_quantity": None, "allocated_quantity": allocated,
                "allocation_value": allocation_value,
                "days_to_cancellation": (cancellation_date - sale_date).days,
                "cancellation_status": None, "unmatched_reason": None,
                "eligible_prior_sales_quantity": None,
            })
            lot["remaining_quantity"] -= allocated
            remaining_request -= allocated
            matched += allocated
        unmatched = requested - matched
        if unmatched == 0:
            status, reason = "MATCHED", None
        elif matched > 0:
            status, reason = "PARTIALLY_MATCHED", "INSUFFICIENT_PRIOR_QUANTITY"
        elif prior_sales_quantity == 0:
            status, reason = "UNMATCHED", "NO_PRIOR_ELIGIBLE_SALE"
        else:
            status, reason = "UNMATCHED", "NO_AVAILABLE_PRIOR_QUANTITY"
        records.append({
            "record_type": "STATUS", "allocation_id": None,
            "cancellation_row_id": event["row_id"], "sale_row_id": None,
            "cancellation_invoice": event["invoice"], "sale_invoice": None,
            "customer_id": event["customer_id"],
            "sale_stock_code": None,
            "cancellation_stock_code": event["stock_code"],
            "unit_price": Decimal(str(event["price"])),
            "sale_date": None,
            "cancellation_date": pd.Timestamp(event["invoice_date"]).to_pydatetime(),
            "requested_quantity": requested, "matched_quantity": matched,
            "unmatched_quantity": unmatched, "allocated_quantity": None,
            "allocation_value": None, "days_to_cancellation": None,
            "cancellation_status": status, "unmatched_reason": reason,
            "eligible_prior_sales_quantity": prior_sales_quantity,
        })
    result = pd.DataFrame(records, columns=_fifo_cols)
    for column in _fifo_int_cols:
        result[column] = pd.array(result[column], dtype="Int64")
    for column in _fifo_ts_cols:
        result[column] = pd.to_datetime(result[column], errors="coerce")
    return result


# ------------------------------------------------------------
# 1. Re-run FIFO allocation
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 6 DELIVERABLE B: Idempotency Test")
print("=" * 70)

ledger = spark.read.parquet(LEDGER_PATH)
cancellations = ledger.filter(F.col("transaction_type") == "CANCELLATION")

t0 = time.time()

matchable_events = (
    ledger.filter(F.col("customer_id").isNotNull())
    .withColumn("_match_stock_code", F.upper(F.trim(F.col("stock_code"))))
    .select(
        "row_id", "invoice", "invoice_date", "customer_id",
        "stock_code", "_match_stock_code", "price", "quantity",
        "transaction_type",
    )
)

grouped_results = (
    matchable_events
    .groupBy("customer_id", "_match_stock_code", "price")
    .applyInPandas(_fifo_group, schema=_fifo_schema)
)

null_customer_status = (
    cancellations.filter(F.col("customer_id").isNull())
    .select(
        F.lit("STATUS").alias("record_type"),
        F.lit(None).cast("string").alias("allocation_id"),
        F.col("row_id").alias("cancellation_row_id"),
        F.lit(None).cast("string").alias("sale_row_id"),
        F.col("invoice").alias("cancellation_invoice"),
        F.lit(None).cast("string").alias("sale_invoice"),
        F.col("customer_id").cast("string").alias("customer_id"),
        F.lit(None).cast("string").alias("sale_stock_code"),
        F.col("stock_code").alias("cancellation_stock_code"),
        F.col("price").cast("decimal(18,4)").alias("unit_price"),
        F.lit(None).cast("timestamp").alias("sale_date"),
        F.col("invoice_date").cast("timestamp").alias("cancellation_date"),
        F.abs(F.col("quantity").cast("long")).alias("requested_quantity"),
        F.lit(0).cast("long").alias("matched_quantity"),
        F.abs(F.col("quantity").cast("long")).alias("unmatched_quantity"),
        F.lit(None).cast("long").alias("allocated_quantity"),
        F.lit(None).cast("decimal(30,4)").alias("allocation_value"),
        F.lit(None).cast("int").alias("days_to_cancellation"),
        F.lit("UNMATCHED").alias("cancellation_status"),
        F.lit("MISSING_CUSTOMER_ID").alias("unmatched_reason"),
        F.lit(0).cast("long").alias("eligible_prior_sales_quantity"),
    )
)

all_results_raw = grouped_results.unionByName(null_customer_status)

(all_results_raw.write.format("parquet").mode("overwrite")
    .option("compression", "snappy").save(TEST_STAGING))

all_results = spark.read.parquet(TEST_STAGING)

test_allocations = all_results.filter(F.col("record_type") == "ALLOCATION").select(
    "allocation_id", "cancellation_row_id", "sale_row_id",
    "cancellation_invoice", "sale_invoice", "customer_id",
    "sale_stock_code", "cancellation_stock_code", "unit_price",
    "sale_date", "cancellation_date", "allocated_quantity",
    "allocation_value", "days_to_cancellation",
)

test_status = all_results.filter(F.col("record_type") == "STATUS").select(
    "cancellation_row_id", "cancellation_invoice", "cancellation_date",
    "customer_id", "cancellation_stock_code", "unit_price",
    "requested_quantity", "matched_quantity", "unmatched_quantity",
    "cancellation_status", "unmatched_reason", "eligible_prior_sales_quantity",
)

test_alloc_count = test_allocations.count()
test_status_count = test_status.count()
fifo_elapsed = time.time() - t0

print(f"\nFIFO rerun: {test_alloc_count:,} allocations, {test_status_count:,} statuses")
print(f"FIFO rerun elapsed: {fifo_elapsed:.1f}s")

# ------------------------------------------------------------
# 2. Re-run financial reporting
# ------------------------------------------------------------

t1 = time.time()

test_inv_summ = (
    ledger.groupBy("invoice").agg(
        F.min("invoice_date").alias("invoice_date"),
        F.max("country").alias("country"),
        F.sum(F.when(F.col("transaction_type") == "SALE", F.col("signed_line_value")).otherwise(ZERO_DEC)).alias("gross_sales_value"),
        F.sum(F.when(F.col("transaction_type") == "CANCELLATION", F.abs(F.col("signed_line_value"))).otherwise(ZERO_DEC)).alias("cancellation_value"),
        F.sum(F.when(F.col("transaction_type") == "SALE", F.lit(1)).otherwise(F.lit(0))).alias("sales_line_count"),
        F.sum(F.when(F.col("transaction_type") == "CANCELLATION", F.lit(1)).otherwise(F.lit(0))).alias("cancellation_line_count"),
        F.sum(F.when(F.col("transaction_type") == "SALE", F.col("quantity").cast("long")).otherwise(F.lit(0))).alias("gross_sold_quantity"),
        F.sum(F.when(F.col("transaction_type") == "CANCELLATION", F.abs(F.col("quantity").cast("long"))).otherwise(F.lit(0))).alias("abs_cancelled_quantity"),
    )
    .withColumn("net_sales_value", (F.col("gross_sales_value") - F.col("cancellation_value")).cast("decimal(30,4)"))
)

test_inv_summ_count = test_inv_summ.count()
report_elapsed = time.time() - t1
print(f"Invoice summary rerun: {test_inv_summ_count:,} records")
print(f"Reporting rerun elapsed: {report_elapsed:.1f}s")

# ------------------------------------------------------------
# 3. Compute deterministic fingerprints for comparison
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("Deterministic fingerprint comparison")
print("-" * 70)

# Allocation fingerprint: sorted (allocation_id, allocated_quantity)
orig_alloc = spark.read.parquet(ORIG_ALLOC)

def alloc_fingerprint(df):
    """Compute a deterministic hash over allocation key columns."""
    sorted_pd = (
        df.select("allocation_id", "allocated_quantity", "cancellation_row_id", "sale_row_id")
        .orderBy("allocation_id")
        .toPandas()
    )
    import hashlib
    data_str = sorted_pd.to_csv(index=False)
    return hashlib.md5(data_str.encode()).hexdigest()

orig_alloc_fp = alloc_fingerprint(orig_alloc)
test_alloc_fp = alloc_fingerprint(test_allocations)
print(f"Allocation fingerprint (original): {orig_alloc_fp}")
print(f"Allocation fingerprint (rerun):     {test_alloc_fp}")
assert orig_alloc_fp == test_alloc_fp, "Allocation fingerprints differ!"
print("PASS: Allocation fingerprints match")

# Allocation count and total
orig_alloc_count = orig_alloc.count()
orig_alloc_total = orig_alloc.agg(F.sum("allocated_quantity")).first()[0]
test_alloc_total = test_allocations.agg(F.sum("allocated_quantity")).first()[0]
assert orig_alloc_count == test_alloc_count, (
    f"Allocation count mismatch: {orig_alloc_count} vs {test_alloc_count}"
)
assert orig_alloc_total == test_alloc_total, (
    f"Allocation total mismatch: {orig_alloc_total} vs {test_alloc_total}"
)
print(f"PASS: Allocation count ({orig_alloc_count:,}) and total ({orig_alloc_total:,}) match")

# Status fingerprint: sorted (cancellation_row_id, cancellation_status, matched, unmatched)
orig_status = spark.read.parquet(ORIG_STATUS)

def status_fingerprint(df):
    sorted_pd = (
        df.select(
            "cancellation_row_id", "cancellation_status",
            "matched_quantity", "unmatched_quantity",
        )
        .orderBy("cancellation_row_id")
        .toPandas()
    )
    import hashlib
    data_str = sorted_pd.to_csv(index=False)
    return hashlib.md5(data_str.encode()).hexdigest()

orig_status_fp = status_fingerprint(orig_status)
test_status_fp = status_fingerprint(test_status)
print(f"\nStatus fingerprint (original): {orig_status_fp}")
print(f"Status fingerprint (rerun):     {test_status_fp}")
assert orig_status_fp == test_status_fp, "Status fingerprints differ!"
print("PASS: Status fingerprints match")

# Status count
orig_status_count = orig_status.count()
assert orig_status_count == test_status_count, (
    f"Status count mismatch: {orig_status_count} vs {test_status_count}"
)
print(f"PASS: Status count ({orig_status_count:,}) matches")

# Invoice summary fingerprint
orig_inv_summ = spark.read.parquet(ORIG_INV_SUMM)

def inv_summ_fingerprint(df):
    sorted_pd = (
        df.select("invoice", "gross_sales_value", "cancellation_value", "net_sales_value")
        .orderBy("invoice")
        .toPandas()
    )
    import hashlib
    data_str = sorted_pd.to_csv(index=False)
    return hashlib.md5(data_str.encode()).hexdigest()

orig_inv_fp = inv_summ_fingerprint(orig_inv_summ)
test_inv_fp = inv_summ_fingerprint(test_inv_summ)
print(f"\nInvoice summary fingerprint (original): {orig_inv_fp}")
print(f"Invoice summary fingerprint (rerun):     {test_inv_fp}")
assert orig_inv_fp == test_inv_fp, "Invoice summary fingerprints differ!"
print("PASS: Invoice summary fingerprints match")

# Financial totals match
orig_gross = orig_inv_summ.agg(F.sum("gross_sales_value")).first()[0]
test_gross = test_inv_summ.agg(F.sum("gross_sales_value")).first()[0]
orig_cancel = orig_inv_summ.agg(F.sum("cancellation_value")).first()[0]
test_cancel = test_inv_summ.agg(F.sum("cancellation_value")).first()[0]
assert orig_gross == test_gross, f"Gross mismatch: {orig_gross} vs {test_gross}"
assert orig_cancel == test_cancel, f"Cancel mismatch: {orig_cancel} vs {test_cancel}"
print(f"PASS: Financial totals match (gross={orig_gross:,.2f}, cancel={orig_cancel:,.2f})")

# ------------------------------------------------------------
# 4. Cleanup test artifacts
# ------------------------------------------------------------

for path in [TEST_STAGING, TEST_ALLOC, TEST_STATUS, TEST_INV_SUMM, TEST_MC]:
    try:
        dbutils.fs.rm(path, recurse=True)
    except Exception:
        pass

print("\n" + "=" * 70)
print("DELIVERABLE B: IDEMPOTENCY TEST PASSED")
print("=" * 70)
print("All deterministic fingerprints match between original and rerun outputs.")
print("No records duplicated or financial totals inflated.")
print(f"\nTiming: FIFO={fifo_elapsed:.1f}s, Reporting={report_elapsed:.1f}s")


In [0]:
# ============================================================
# RetailRecon | Phase 6: Deliverable C
# Spark Performance Benchmark
# ============================================================
# Compares baseline FIFO (all groups through applyInPandas)
# with an optimized pre-filtered version (only groups with
# at least one cancellation).
# Correctness equivalence is validated before recommending.
# ============================================================

from collections import deque
from decimal import Decimal
from hashlib import sha256
import time
import hashlib

import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DecimalType, LongType, IntegerType,
)

BASE_PATH = "/Volumes/portfolio_projects/retail_sales/raw"
LEDGER_PATH = f"{BASE_PATH}/classified_sales_ledger"
BENCH_STAGING = f"{BASE_PATH}/_bench_staging"
BENCH_STAGING_OPT = f"{BASE_PATH}/_bench_staging_opt"

ORIG_ALLOC = f"{BASE_PATH}/cancellation_allocations_parquet"
ORIG_STATUS = f"{BASE_PATH}/cancellation_status_parquet"

print("=" * 70)
print("PHASE 6 DELIVERABLE C: Performance Benchmark")
print("=" * 70)

ledger = spark.read.parquet(LEDGER_PATH)
cancellations = ledger.filter(F.col("transaction_type") == "CANCELLATION")

# ------------------------------------------------------------
# Self-contained FIFO function (identical to cell 10)
# ------------------------------------------------------------

_fifo_schema = StructType([
    StructField("record_type", StringType(), True),
    StructField("allocation_id", StringType(), True),
    StructField("cancellation_row_id", StringType(), True),
    StructField("sale_row_id", StringType(), True),
    StructField("cancellation_invoice", StringType(), True),
    StructField("sale_invoice", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("sale_stock_code", StringType(), True),
    StructField("cancellation_stock_code", StringType(), True),
    StructField("unit_price", DecimalType(18, 4), True),
    StructField("sale_date", TimestampType(), True),
    StructField("cancellation_date", TimestampType(), True),
    StructField("requested_quantity", LongType(), True),
    StructField("matched_quantity", LongType(), True),
    StructField("unmatched_quantity", LongType(), True),
    StructField("allocated_quantity", LongType(), True),
    StructField("allocation_value", DecimalType(30, 4), True),
    StructField("days_to_cancellation", IntegerType(), True),
    StructField("cancellation_status", StringType(), True),
    StructField("unmatched_reason", StringType(), True),
    StructField("eligible_prior_sales_quantity", LongType(), True),
])

_fifo_cols = [f.name for f in _fifo_schema.fields]
_fifo_int_cols = [
    "requested_quantity", "matched_quantity", "unmatched_quantity",
    "allocated_quantity", "days_to_cancellation",
    "eligible_prior_sales_quantity",
]
_fifo_ts_cols = ["sale_date", "cancellation_date"]


def _fifo_group(pdf):
    if pdf.empty:
        return pd.DataFrame(columns=_fifo_cols)
    pdf = pdf.copy()
    pdf["_event_priority"] = pdf["transaction_type"].map(
        {"CANCELLATION": 0, "SALE": 1}
    )
    pdf = pdf.sort_values(
        by=["invoice_date", "_event_priority", "invoice", "row_id"],
        ascending=True, kind="mergesort",
    )
    inventory = deque()
    records = []
    prior_sales_quantity = 0
    for event in pdf.to_dict("records"):
        event_type = event["transaction_type"]
        if event_type == "SALE":
            quantity = int(event["quantity"])
            if quantity <= 0:
                raise ValueError(f"Invalid sale quantity for row {event['row_id']}")
            inventory.append({
                "row_id": event["row_id"], "invoice": event["invoice"],
                "invoice_date": event["invoice_date"],
                "stock_code": event["stock_code"],
                "price": event["price"],
                "remaining_quantity": quantity,
            })
            prior_sales_quantity += quantity
            continue
        requested = abs(int(event["quantity"]))
        if requested <= 0:
            raise ValueError(f"Invalid cancellation quantity for row {event['row_id']}")
        remaining_request = requested
        matched = 0
        while remaining_request > 0 and inventory:
            while inventory and inventory[0]["remaining_quantity"] == 0:
                inventory.popleft()
            if not inventory:
                break
            lot = inventory[0]
            allocated = min(remaining_request, lot["remaining_quantity"])
            if allocated <= 0:
                raise ValueError("Non-positive FIFO allocation detected.")
            price = Decimal(str(event["price"]))
            allocation_value = (price * Decimal(allocated)).quantize(Decimal("0.0001"))
            sale_date = pd.Timestamp(lot["invoice_date"])
            cancellation_date = pd.Timestamp(event["invoice_date"])
            if sale_date >= cancellation_date:
                raise ValueError("FIFO attempted to match a sale at or after cancellation")
            pair_key = f"{lot['row_id']}||{event['row_id']}"
            records.append({
                "record_type": "ALLOCATION",
                "allocation_id": sha256(pair_key.encode("utf-8")).hexdigest(),
                "cancellation_row_id": event["row_id"],
                "sale_row_id": lot["row_id"],
                "cancellation_invoice": event["invoice"],
                "sale_invoice": lot["invoice"],
                "customer_id": event["customer_id"],
                "sale_stock_code": lot["stock_code"],
                "cancellation_stock_code": event["stock_code"],
                "unit_price": price,
                "sale_date": sale_date.to_pydatetime(),
                "cancellation_date": cancellation_date.to_pydatetime(),
                "requested_quantity": None, "matched_quantity": None,
                "unmatched_quantity": None, "allocated_quantity": allocated,
                "allocation_value": allocation_value,
                "days_to_cancellation": (cancellation_date - sale_date).days,
                "cancellation_status": None, "unmatched_reason": None,
                "eligible_prior_sales_quantity": None,
            })
            lot["remaining_quantity"] -= allocated
            remaining_request -= allocated
            matched += allocated
        unmatched = requested - matched
        if unmatched == 0:
            status, reason = "MATCHED", None
        elif matched > 0:
            status, reason = "PARTIALLY_MATCHED", "INSUFFICIENT_PRIOR_QUANTITY"
        elif prior_sales_quantity == 0:
            status, reason = "UNMATCHED", "NO_PRIOR_ELIGIBLE_SALE"
        else:
            status, reason = "UNMATCHED", "NO_AVAILABLE_PRIOR_QUANTITY"
        records.append({
            "record_type": "STATUS", "allocation_id": None,
            "cancellation_row_id": event["row_id"], "sale_row_id": None,
            "cancellation_invoice": event["invoice"], "sale_invoice": None,
            "customer_id": event["customer_id"],
            "sale_stock_code": None,
            "cancellation_stock_code": event["stock_code"],
            "unit_price": Decimal(str(event["price"])),
            "sale_date": None,
            "cancellation_date": pd.Timestamp(event["invoice_date"]).to_pydatetime(),
            "requested_quantity": requested, "matched_quantity": matched,
            "unmatched_quantity": unmatched, "allocated_quantity": None,
            "allocation_value": None, "days_to_cancellation": None,
            "cancellation_status": status, "unmatched_reason": reason,
            "eligible_prior_sales_quantity": prior_sales_quantity,
        })
    result = pd.DataFrame(records, columns=_fifo_cols)
    for column in _fifo_int_cols:
        result[column] = pd.array(result[column], dtype="Int64")
    for column in _fifo_ts_cols:
        result[column] = pd.to_datetime(result[column], errors="coerce")
    return result


def _make_null_status(cancellations_df):
    return (
        cancellations_df.filter(F.col("customer_id").isNull())
        .select(
            F.lit("STATUS").alias("record_type"),
            F.lit(None).cast("string").alias("allocation_id"),
            F.col("row_id").alias("cancellation_row_id"),
            F.lit(None).cast("string").alias("sale_row_id"),
            F.col("invoice").alias("cancellation_invoice"),
            F.lit(None).cast("string").alias("sale_invoice"),
            F.col("customer_id").cast("string").alias("customer_id"),
            F.lit(None).cast("string").alias("sale_stock_code"),
            F.col("stock_code").alias("cancellation_stock_code"),
            F.col("price").cast("decimal(18,4)").alias("unit_price"),
            F.lit(None).cast("timestamp").alias("sale_date"),
            F.col("invoice_date").cast("timestamp").alias("cancellation_date"),
            F.abs(F.col("quantity").cast("long")).alias("requested_quantity"),
            F.lit(0).cast("long").alias("matched_quantity"),
            F.abs(F.col("quantity").cast("long")).alias("unmatched_quantity"),
            F.lit(None).cast("long").alias("allocated_quantity"),
            F.lit(None).cast("decimal(30,4)").alias("allocation_value"),
            F.lit(None).cast("int").alias("days_to_cancellation"),
            F.lit("UNMATCHED").alias("cancellation_status"),
            F.lit("MISSING_CUSTOMER_ID").alias("unmatched_reason"),
            F.lit(0).cast("long").alias("eligible_prior_sales_quantity"),
        )
    )


def _alloc_fingerprint(df):
    sorted_pd = (
        df.select("allocation_id", "allocated_quantity", "cancellation_row_id", "sale_row_id")
        .orderBy("allocation_id")
        .toPandas()
    )
    return hashlib.md5(sorted_pd.to_csv(index=False).encode()).hexdigest()


def _status_fingerprint(df):
    sorted_pd = (
        df.select("cancellation_row_id", "cancellation_status", "matched_quantity", "unmatched_quantity")
        .orderBy("cancellation_row_id")
        .toPandas()
    )
    return hashlib.md5(sorted_pd.to_csv(index=False).encode()).hexdigest()

# ------------------------------------------------------------
# 1. Analyze group distribution (bottleneck identification)
# ------------------------------------------------------------

print("\n--- Group distribution analysis ---")

matchable = (
    ledger.filter(F.col("customer_id").isNotNull())
    .withColumn("_match_stock_code", F.upper(F.trim(F.col("stock_code"))))
    .select(
        "row_id", "invoice", "invoice_date", "customer_id",
        "stock_code", "_match_stock_code", "price", "quantity",
        "transaction_type",
    )
)

group_counts = (
    matchable
    .groupBy("customer_id", "_match_stock_code", "price")
    .agg(
        F.sum(F.when(F.col("transaction_type") == "SALE", F.lit(1)).otherwise(F.lit(0))).alias("sale_rows"),
        F.sum(F.when(F.col("transaction_type") == "CANCELLATION", F.lit(1)).otherwise(F.lit(0))).alias("cancel_rows"),
    )
)

total_groups = group_counts.count()
groups_with_cancel = group_counts.filter(F.col("cancel_rows") > 0).count()
groups_sales_only = group_counts.filter(F.col("cancel_rows") == 0).count()

matchable_rows = matchable.count()
rows_with_cancel = (
    matchable.join(
        group_counts.filter(F.col("cancel_rows") > 0)
        .select("customer_id", "_match_stock_code", "price"),
        ["customer_id", "_match_stock_code", "price"],
        "inner",
    ).count()
)

print(f"Total matching groups:           {total_groups:,}")
print(f"Groups with cancellations:       {groups_with_cancel:,}")
print(f"Groups with sales only (no canc): {groups_sales_only:,}")
print(f"Groups with cancellations ratio: {groups_with_cancel / total_groups:.1%}")
print(f"Total matchable rows:             {matchable_rows:,}")
print(f"Rows in groups with cancellations: {rows_with_cancel:,}")
print(f"Rows filtered out (no cancellations): {matchable_rows - rows_with_cancel:,}")
print(f"Row reduction ratio: {(matchable_rows - rows_with_cancel) / matchable_rows:.1%}")

# ------------------------------------------------------------
# 2. Baseline run (all groups through applyInPandas)
# ------------------------------------------------------------

print("\n--- Baseline run (all groups) ---")
t0 = time.time()

baseline_grouped = (
    matchable
    .groupBy("customer_id", "_match_stock_code", "price")
    .applyInPandas(_fifo_group, schema=_fifo_schema)
)

baseline_all = baseline_grouped.unionByName(_make_null_status(cancellations))

(baseline_all.write.format("parquet").mode("overwrite")
    .option("compression", "snappy").save(BENCH_STAGING))

baseline_results = spark.read.parquet(BENCH_STAGING)
baseline_alloc = baseline_results.filter(F.col("record_type") == "ALLOCATION").select(
    "allocation_id", "cancellation_row_id", "sale_row_id",
    "cancellation_invoice", "sale_invoice", "customer_id",
    "sale_stock_code", "cancellation_stock_code", "unit_price",
    "sale_date", "cancellation_date", "allocated_quantity",
    "allocation_value", "days_to_cancellation",
)
baseline_status = baseline_results.filter(F.col("record_type") == "STATUS").select(
    "cancellation_row_id", "cancellation_invoice", "cancellation_date",
    "customer_id", "cancellation_stock_code", "unit_price",
    "requested_quantity", "matched_quantity", "unmatched_quantity",
    "cancellation_status", "unmatched_reason", "eligible_prior_sales_quantity",
)

baseline_alloc_count = baseline_alloc.count()
baseline_status_count = baseline_status.count()
baseline_elapsed = time.time() - t0

print(f"Baseline: {baseline_alloc_count:,} allocations, {baseline_status_count:,} statuses")
print(f"Baseline elapsed: {baseline_elapsed:.1f}s")

# ------------------------------------------------------------
# 3. Optimized run (pre-filter to groups with cancellations only)
# ------------------------------------------------------------

print("\n--- Optimized run (pre-filtered groups) ---")

cancel_group_keys = (
    matchable
    .filter(F.col("transaction_type") == "CANCELLATION")
    .select("customer_id", "_match_stock_code", "price")
    .distinct()
)

optimized_matchable = matchable.join(
    cancel_group_keys,
    ["customer_id", "_match_stock_code", "price"],
    "inner",
)

optimized_row_count = optimized_matchable.count()
print(f"Optimized input rows: {optimized_row_count:,} (from {matchable_rows:,})")

t1 = time.time()

optimized_grouped = (
    optimized_matchable
    .groupBy("customer_id", "_match_stock_code", "price")
    .applyInPandas(_fifo_group, schema=_fifo_schema)
)

optimized_all = optimized_grouped.unionByName(_make_null_status(cancellations))

(optimized_all.write.format("parquet").mode("overwrite")
    .option("compression", "snappy").save(BENCH_STAGING_OPT))

optimized_results = spark.read.parquet(BENCH_STAGING_OPT)
optimized_alloc = optimized_results.filter(F.col("record_type") == "ALLOCATION").select(
    "allocation_id", "cancellation_row_id", "sale_row_id",
    "cancellation_invoice", "sale_invoice", "customer_id",
    "sale_stock_code", "cancellation_stock_code", "unit_price",
    "sale_date", "cancellation_date", "allocated_quantity",
    "allocation_value", "days_to_cancellation",
)
optimized_status = optimized_results.filter(F.col("record_type") == "STATUS").select(
    "cancellation_row_id", "cancellation_invoice", "cancellation_date",
    "customer_id", "cancellation_stock_code", "unit_price",
    "requested_quantity", "matched_quantity", "unmatched_quantity",
    "cancellation_status", "unmatched_reason", "eligible_prior_sales_quantity",
)

optimized_alloc_count = optimized_alloc.count()
optimized_status_count = optimized_status.count()
optimized_elapsed = time.time() - t1

print(f"Optimized: {optimized_alloc_count:,} allocations, {optimized_status_count:,} statuses")
print(f"Optimized elapsed: {optimized_elapsed:.1f}s")

# ------------------------------------------------------------
# 4. Correctness equivalence validation
# ------------------------------------------------------------

print("\n--- Correctness equivalence validation ---")

# Compare allocation fingerprints
baseline_alloc_fp = _alloc_fingerprint(baseline_alloc)
optimized_alloc_fp = _alloc_fingerprint(optimized_alloc)
print(f"Baseline allocation FP:  {baseline_alloc_fp}")
print(f"Optimized allocation FP: {optimized_alloc_fp}")
assert baseline_alloc_fp == optimized_alloc_fp, "Allocation fingerprints differ!"
print("PASS: Allocation fingerprints match")

# Compare status fingerprints
baseline_status_fp = _status_fingerprint(baseline_status)
optimized_status_fp = _status_fingerprint(optimized_status)
print(f"Baseline status FP:  {baseline_status_fp}")
print(f"Optimized status FP: {optimized_status_fp}")
assert baseline_status_fp == optimized_status_fp, "Status fingerprints differ!"
print("PASS: Status fingerprints match")

# Compare with original persisted outputs
orig_alloc = spark.read.parquet(ORIG_ALLOC)
orig_status = spark.read.parquet(ORIG_STATUS)
orig_alloc_fp = _alloc_fingerprint(orig_alloc)
orig_status_fp = _status_fingerprint(orig_status)
assert orig_alloc_fp == optimized_alloc_fp, "Optimized differs from original!"
assert orig_status_fp == optimized_status_fp, "Optimized status differs from original!"
print("PASS: Both implementations match original persisted outputs")

# ------------------------------------------------------------
# 5. Summary report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DELIVERABLE C: PERFORMANCE BENCHMARK SUMMARY")
print("=" * 70)

speedup = baseline_elapsed / optimized_elapsed if optimized_elapsed > 0 else float('inf')

print(f"\n| Metric                    | Baseline        | Optimized       |")
print(f"|---------------------------|-----------------|-----------------|")
print(f"| Input rows to applyInPandas| {matchable_rows:>15,} | {optimized_row_count:>15,} |")
print(f"| Groups processed           | {total_groups:>15,} | {groups_with_cancel:>15,} |")
print(f"| Allocation count          | {baseline_alloc_count:>15,} | {optimized_alloc_count:>15,} |")
print(f"| Status count              | {baseline_status_count:>15,} | {optimized_status_count:>15,} |")
print(f"| Elapsed time (s)          | {baseline_elapsed:>15.1f} | {optimized_elapsed:>15.1f} |")
print(f"| Speedup                   | {'1.0x (ref)':>15} | {speedup:>14.2f}x |")

print(f"\nOptimization: Pre-filter to groups containing at least one cancellation.")
print(f"Rationale: {groups_sales_only:,} of {total_groups:,} groups ({groups_sales_only/total_groups:.1%}) have sales only")
print(f"and produce zero allocation/status records. Pre-filtering removes")
print(f"{matchable_rows - rows_with_cancel:,} of {matchable_rows:,} rows ({(matchable_rows - rows_with_cancel)/matchable_rows:.1%})")
print(f"before the expensive Python serialization stage.")

print(f"\nBoth implementations produce identical allocation and status outputs.")
print(f"The optimized version is recommended for future pipeline runs.")

# Cleanup benchmark artifacts
for path in [BENCH_STAGING, BENCH_STAGING_OPT]:
    try:
        dbutils.fs.rm(path, recurse=True)
    except Exception:
        pass

print("\nBenchmark artifacts cleaned up.")


In [0]:
# ============================================================
# RetailRecon | Phase 7: Final End-to-End Validation
# Comprehensive A+B+C checks against persisted outputs.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE_PATH = "/Volumes/portfolio_projects/retail_sales/raw"

RAW_PATH = f"{BASE_PATH}/source_raw_parquet"
TYPED_PATH = f"{BASE_PATH}/source_typed_parquet"
EXCLUSIONS_PATH = f"{BASE_PATH}/source_overlap_exclusions_parquet"
LEDGER_PATH = f"{BASE_PATH}/classified_sales_ledger"
ADJUSTMENTS_PATH = f"{BASE_PATH}/classified_adjustments"
EXCEPTIONS_PATH = f"{BASE_PATH}/classified_exceptions"
ALLOCATIONS_PATH = f"{BASE_PATH}/cancellation_allocations_parquet"
STATUS_PATH = f"{BASE_PATH}/cancellation_status_parquet"
INVOICE_SUMMARY_PATH = f"{BASE_PATH}/invoice_summary_parquet"
MONTHLY_COUNTRY_PATH = f"{BASE_PATH}/monthly_country_sales_parquet"
CUSTOMER_HISTORY_PATH = f"{BASE_PATH}/customer_purchase_history_parquet"

ZERO_DEC = F.lit(0).cast("decimal(30,4)")

# --- Read all persisted datasets ---
raw = spark.read.parquet(RAW_PATH)
typed = spark.read.parquet(TYPED_PATH)
exclusions = spark.read.parquet(EXCLUSIONS_PATH)
ledger = spark.read.parquet(LEDGER_PATH)
adjustments = spark.read.parquet(ADJUSTMENTS_PATH)
exceptions = spark.read.parquet(EXCEPTIONS_PATH)
allocations = spark.read.parquet(ALLOCATIONS_PATH)
status = spark.read.parquet(STATUS_PATH)
invoice_summary = spark.read.parquet(INVOICE_SUMMARY_PATH)
monthly_country = spark.read.parquet(MONTHLY_COUNTRY_PATH)
customer_history = spark.read.parquet(CUSTOMER_HISTORY_PATH)

# --- Compute all counts ---
raw_count = raw.count()
typed_count = typed.count()
exclusion_count = exclusions.count()
ledger_count = ledger.count()
adjustment_count = adjustments.count()
exception_count = exceptions.count()
sales_count = ledger.filter(F.col("transaction_type") == "SALE").count()
cancel_count = ledger.filter(F.col("transaction_type") == "CANCELLATION").count()
alloc_count = allocations.count()
status_count = status.count()
inv_summ_count = invoice_summary.count()
mc_count = monthly_country.count()
ch_count = customer_history.count()

gross_total = invoice_summary.agg(F.sum("gross_sales_value")).first()[0]
cancel_total = invoice_summary.agg(F.sum("cancellation_value")).first()[0]
net_total = gross_total - cancel_total

allocated_qty = allocations.agg(F.coalesce(F.sum("allocated_quantity"), F.lit(0).cast("long")).alias("t")).first()["t"]
matched_qty = status.agg(F.coalesce(F.sum("matched_quantity"), F.lit(0).cast("long")).alias("t")).first()["t"]
unmatched_qty = status.agg(F.coalesce(F.sum("unmatched_quantity"), F.lit(0).cast("long")).alias("t")).first()["t"]
requested_qty = status.agg(F.coalesce(F.sum("requested_quantity"), F.lit(0).cast("long")).alias("t")).first()["t"]
distinct_customers = customer_history.select("customer_id").distinct().count()

# ============================================================
# TEST TRACKING
# ============================================================
results = []

def record(test_id, description, passed, actual=None, expected=None):
    status_str = "PASS" if passed else "FAIL"
    results.append((test_id, description, status_str, actual, expected))
    if passed:
        print(f"  [{status_str}] {test_id}: {description}")
    else:
        print(f"  [{status_str}] {test_id}: {description} | actual={actual} expected={expected}")

print("=" * 70)
print("PHASE 7: FINAL END-TO-END VALIDATION")
print("=" * 70)

# ============================================================
# A. SOURCE ACCOUNTING
# ============================================================
print("\n--- A. SOURCE ACCOUNTING ---")

record("A1", "Raw rows = 1,067,371",
       raw_count == 1_067_371, raw_count, 1_067_371)
record("A2", "Overlap exclusions = 22,523",
       exclusion_count == 22_523, exclusion_count, 22_523)
record("A3", "Retained typed rows = 1,044,848",
       typed_count == 1_044_848, typed_count, 1_044_848)
record("A4", "Merchandise ledger = 1,033,442",
       ledger_count == 1_033_442, ledger_count, 1_033_442)
record("A5", "Adjustments = 5,377",
       adjustment_count == 5_377, adjustment_count, 5_377)
record("A6", "Exceptions = 6,029",
       exception_count == 6_029, exception_count, 6_029)
record("A7", "Sales = 1,015,251",
       sales_count == 1_015_251, sales_count, 1_015_251)
record("A8", "Cancellations = 18,191",
       cancel_count == 18_191, cancel_count, 18_191)

record("A9", "Conservation: raw = typed + exclusions",
       raw_count == typed_count + exclusion_count,
       f"{raw_count} = {typed_count} + {exclusion_count}",
       f"{typed_count + exclusion_count}")
record("A10", "Conservation: typed = ledger + adjustments + exceptions",
        typed_count == ledger_count + adjustment_count + exception_count,
        f"{typed_count} = {ledger_count} + {adjustment_count} + {exception_count}",
        f"{ledger_count + adjustment_count + exception_count}")
record("A11", "Conservation: sales + cancellations = ledger",
        sales_count + cancel_count == ledger_count,
        f"{sales_count} + {cancel_count} = {ledger_count}",
        f"{ledger_count}")

# Row-ID uniqueness
raw_dup = raw.count() - raw.select("row_id").distinct().count()
typed_dup = typed_count - typed.select("row_id").distinct().count()
ledger_dup = ledger_count - ledger.select("row_id").distinct().count()
record("A12", "Raw row_id uniqueness", raw_dup == 0, f"dups={raw_dup}", "dups=0")
record("A13", "Typed row_id uniqueness", typed_dup == 0, f"dups={typed_dup}", "dups=0")
record("A14", "Ledger row_id uniqueness", ledger_dup == 0, f"dups={ledger_dup}", "dups=0")

# Classification exclusivity
all_ids = (ledger.select("row_id")
           .unionByName(adjustments.select("row_id"))
           .unionByName(exceptions.select("row_id")))
total_ids = all_ids.count()
unique_ids = all_ids.distinct().count()
record("A15", "Classification exclusivity (total=unique=typed)",
        total_ids == unique_ids == typed_count,
        f"total={total_ids}, unique={unique_ids}, typed={typed_count}",
        f"all={typed_count}")

# No null row_ids
null_raw = raw.filter(F.col("row_id").isNull()).limit(1).count()
null_ledger = ledger.filter(F.col("row_id").isNull()).limit(1).count()
record("A16", "No null row_id in raw", null_raw == 0, f"nulls={null_raw}", "nulls=0")
record("A17", "No null row_id in ledger", null_ledger == 0, f"nulls={null_ledger}", "nulls=0")

# ============================================================
# B. FIFO CORRECTNESS
# ============================================================
print("\n--- B. FIFO CORRECTNESS ---")

record("B1", "Status count = 18,191", status_count == 18_191, status_count, 18_191)
record("B2", "Allocation rows = 16,339", alloc_count == 16_339, alloc_count, 16_339)
record("B3", "Allocated units = 433,303", allocated_qty == 433_303, allocated_qty, 433_303)
record("B4", "Unmatched units = 36,807", unmatched_qty == 36_807, unmatched_qty, 36_807)
record("B5", "Requested units = 470,110", requested_qty == 470_110, requested_qty, 470_110)

# B6: Exactly one status per cancellation
distinct_cancel_ids = status.select("cancellation_row_id").distinct().count()
record("B6", "Exactly one status per cancellation",
        status_count == cancel_count == distinct_cancel_ids,
        f"status={status_count}, cancels={cancel_count}, distinct={distinct_cancel_ids}",
        f"all={cancel_count}")

# B7: requested = matched + unmatched
bad_recon = status.filter(
    F.col("requested_quantity") != F.col("matched_quantity") + F.col("unmatched_quantity")
).limit(1).count()
record("B7", "requested = matched + unmatched", bad_recon == 0, f"bad={bad_recon}", "bad=0")

# B8: No sale allocated beyond original quantity
sale_consumption = allocations.groupBy("sale_row_id").agg(
    F.sum("allocated_quantity").alias("consumed")
)
sale_quantities = ledger.filter(F.col("transaction_type") == "SALE").select(
    F.col("row_id").alias("sale_row_id"),
    F.col("quantity").cast("long").alias("sold"),
)
overalloc = sale_consumption.join(sale_quantities, "sale_row_id", "inner").filter(
    F.col("consumed") > F.col("sold")
).limit(1).count()
record("B8", "No over-allocation", overalloc == 0, f"overalloc={overalloc}", "overalloc=0")

# B9: Sale timestamp strictly earlier than cancellation timestamp
bad_ts = allocations.filter(F.col("sale_date") >= F.col("cancellation_date")).limit(1).count()
record("B9", "Sale timestamp < cancellation timestamp", bad_ts == 0, f"bad={bad_ts}", "bad=0")

# B10: No null-customer cancellation is allocated
null_alloc = allocations.filter(F.col("customer_id").isNull()).limit(1).count()
null_status_matched = status.filter(
    (F.col("customer_id").isNull()) & (F.col("matched_quantity") > 0)
).limit(1).count()
null_missing_count = status.filter(
    (F.col("customer_id").isNull()) & (F.col("unmatched_reason") == "MISSING_CUSTOMER_ID")
).count()
record("B10", "Null-customer cancellations all unmatched",
        null_alloc == 0 and null_status_matched == 0,
        f"null_alloc={null_alloc}, null_matched={null_status_matched}", "all=0")
record("B11", "Null-customer count = 494 with MISSING_CUSTOMER_ID",
        null_missing_count == 494, null_missing_count, 494)

# B12: Allocation IDs unique and deterministic
distinct_alloc_ids = allocations.select("allocation_id").distinct().count()
record("B12", "Allocation IDs unique",
        distinct_alloc_ids == alloc_count,
        f"distinct={distinct_alloc_ids}, total={alloc_count}", "equal")

# B13: Allocated totals reconcile with matched quantities
record("B13", "Allocated total = matched total",
        allocated_qty == matched_qty, f"{allocated_qty} vs {matched_qty}", "equal")

# ============================================================
# C. FINANCIAL REPORTING
# ============================================================
print("\n--- C. FINANCIAL REPORTING ---")

record("C1", "Invoice summaries = 47,155", inv_summ_count == 47_155, inv_summ_count, 47_155)
record("C2", "Monthly-country groups = 585", mc_count == 585, mc_count, 585)
record("C3", "Customer purchase-history rows = 36,669", ch_count == 36_669, ch_count, 36_669)
record("C4", "Distinct identified customers = 5,861",
        distinct_customers == 5_861, distinct_customers, 5_861)

record("C5", "Gross sales = GBP 19,713,221.70",
        abs(float(str(gross_total)) - 19713221.70) < 0.01, str(gross_total), "19713221.70")
record("C6", "Cancellation value = GBP 771,920.31",
        abs(float(str(cancel_total)) - 771920.31) < 0.01, str(cancel_total), "771920.31")
record("C7", "Net sales = GBP 18,941,301.39",
        abs(float(str(net_total)) - 18941301.39) < 0.01, str(net_total), "18941301.39")

# C8: Invoice-to-monthly-country reconciliation
inv_to_monthly = (
    invoice_summary
    .withColumn("invoice_year", F.year("invoice_date"))
    .withColumn("invoice_month", F.month("invoice_date"))
    .withColumn("country_resolved", F.coalesce(F.col("country"), F.lit("UNKNOWN")))
    .groupBy("invoice_year", "invoice_month", "country_resolved")
    .agg(
        F.sum("gross_sales_value").alias("inv_gross"),
        F.sum("cancellation_value").alias("inv_cancel"),
        F.sum("net_sales_value").alias("inv_net"),
    )
)
mc = monthly_country.select(
    F.col("invoice_year"), F.col("invoice_month"),
    F.col("country").alias("country_resolved"),
    F.col("gross_sales_value").alias("mc_gross"),
    F.col("cancellation_value").alias("mc_cancel"),
    F.col("net_sales_value").alias("mc_net"),
)
diff_c8 = (
    inv_to_monthly.join(mc, ["invoice_year", "invoice_month", "country_resolved"], "full_outer")
    .filter(
        (F.coalesce(F.col("inv_gross"), ZERO_DEC) != F.coalesce(F.col("mc_gross"), ZERO_DEC))
        | (F.coalesce(F.col("inv_cancel"), ZERO_DEC) != F.coalesce(F.col("mc_cancel"), ZERO_DEC))
        | (F.coalesce(F.col("inv_net"), ZERO_DEC) != F.coalesce(F.col("mc_net"), ZERO_DEC))
    )
).count()
record("C8", "Invoice-to-monthly-country reconciliation", diff_c8 == 0, f"diffs={diff_c8}", "diffs=0")

# C9: Net-value identity (invoice summary)
bad_inv_net = invoice_summary.filter(
    F.col("net_sales_value") != F.col("gross_sales_value") - F.col("cancellation_value")
).limit(1).count()
record("C9", "Net = gross - cancellation (invoice)", bad_inv_net == 0, f"bad={bad_inv_net}", "bad=0")

# C10: Net-value identity (monthly-country)
bad_mc_net = monthly_country.filter(
    F.col("net_sales_value") != F.col("gross_sales_value") - F.col("cancellation_value")
).limit(1).count()
record("C10", "Net = gross - cancellation (monthly)", bad_mc_net == 0, f"bad={bad_mc_net}", "bad=0")

# C11: Source line-count conservation
inv_sales_lines = invoice_summary.agg(F.sum("sales_line_count")).first()[0]
inv_cancel_lines = invoice_summary.agg(F.sum("cancellation_line_count")).first()[0]
mc_sales_lines = monthly_country.agg(F.sum("sales_line_count")).first()[0]
mc_cancel_lines = monthly_country.agg(F.sum("cancellation_line_count")).first()[0]
record("C11", "Invoice sales lines = ledger sales",
        inv_sales_lines == sales_count, f"{inv_sales_lines} vs {sales_count}", "equal")
record("C12", "Invoice cancel lines = ledger cancellations",
        inv_cancel_lines == cancel_count, f"{inv_cancel_lines} vs {cancel_count}", "equal")
record("C13", "Monthly sales lines = ledger sales",
        mc_sales_lines == sales_count, f"{mc_sales_lines} vs {sales_count}", "equal")
record("C14", "Monthly cancel lines = ledger cancellations",
        mc_cancel_lines == cancel_count, f"{mc_cancel_lines} vs {cancel_count}", "equal")

# C15: Cancellation inclusion (value and quantity)
ledger_cancel_val = ledger.filter(F.col("transaction_type") == "CANCELLATION").agg(
    F.sum(F.abs(F.col("signed_line_value")))
).first()[0]
ledger_cancel_qty = ledger.filter(F.col("transaction_type") == "CANCELLATION").agg(
    F.sum(F.abs(F.col("quantity").cast("long")))
).first()[0]
inv_cancel_val = invoice_summary.agg(F.sum("cancellation_value")).first()[0]
inv_cancel_qty = invoice_summary.agg(F.sum("abs_cancelled_quantity")).first()[0]
mc_cancel_val = monthly_country.agg(F.sum("cancellation_value")).first()[0]
mc_cancel_qty = monthly_country.agg(F.sum("abs_cancelled_quantity")).first()[0]
record("C15", "Invoice cancel value = ledger",
        inv_cancel_val == ledger_cancel_val, str(inv_cancel_val), str(ledger_cancel_val))
record("C16", "Invoice cancel qty = ledger",
        inv_cancel_qty == ledger_cancel_qty, str(inv_cancel_qty), str(ledger_cancel_qty))
record("C17", "Monthly cancel value = ledger",
        mc_cancel_val == ledger_cancel_val, str(mc_cancel_val), str(ledger_cancel_val))
record("C18", "Monthly cancel qty = ledger",
        mc_cancel_qty == ledger_cancel_qty, str(mc_cancel_qty), str(ledger_cancel_qty))

# C19: Customer history — unique keys
ch_dupes = customer_history.groupBy("customer_id", "purchase_invoice").count().filter(
    F.col("count") > 1
).count()
ch_distinct = customer_history.select("customer_id", "purchase_invoice").distinct().count()
record("C19", "Customer history unique keys",
        ch_dupes == 0 and ch_distinct == ch_count,
        f"dupes={ch_dupes}, distinct={ch_distinct}, total={ch_count}", "all unique")

# C20: Customer history — no null customer_id
null_ch = customer_history.filter(F.col("customer_id").isNull()).limit(1).count()
record("C20", "No null customer_id in history", null_ch == 0, f"nulls={null_ch}", "nulls=0")

# C21: Customer history — first purchase date is earliest per customer
w_cust = Window.partitionBy("customer_id")
bad_first = customer_history.withColumn(
    "computed_first", F.min("purchase_invoice_timestamp").over(w_cust)
).filter(F.col("first_purchase_date") != F.col("computed_first")).limit(1).count()
record("C21", "First purchase date = min per customer", bad_first == 0, f"bad={bad_first}", "bad=0")

# C22: Persisted-output schema check (all datasets readable)
schema_ok = (
    len(ledger.columns) == 23
    and len(allocations.columns) == 14
    and len(status.columns) == 12
    and len(invoice_summary.columns) >= 9
    and len(monthly_country.columns) >= 7
    and len(customer_history.columns) >= 9
)
record("C22", "All persisted schemas readable",
        schema_ok,
        f"ledger={len(ledger.columns)}, alloc={len(allocations.columns)}, status={len(status.columns)}, inv={len(invoice_summary.columns)}, mc={len(monthly_country.columns)}, ch={len(customer_history.columns)}",
        "schemas present")

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 70)
print("PHASE 7 VALIDATION SUMMARY")
print("=" * 70)
passed = sum(1 for r in results if r[2] == "PASS")
failed = sum(1 for r in results if r[2] == "FAIL")
print(f"Total tests: {len(results)}")
print(f"Passed: {passed}")
print(f"Failed: {failed}")

if failed > 0:
    print("\nFAILED TESTS:")
    for r in results:
        if r[2] == "FAIL":
            print(f"  {r[0]}: {r[1]} | actual={r[3]} expected={r[4]}")

print(f"\n{'ALL TESTS PASSED' if failed == 0 else 'VALIDATION FAILED'}")
print(f"Row counts: raw={raw_count:,}, typed={typed_count:,}, ledger={ledger_count:,}, sales={sales_count:,}, cancels={cancel_count:,}")
print(f"Financials: gross=GBP {gross_total:,.2f}, cancel=GBP {cancel_total:,.2f}, net=GBP {net_total:,.2f}")
print(f"FIFO: allocations={alloc_count:,}, allocated_qty={allocated_qty:,}, unmatched_qty={unmatched_qty:,}")
print(f"Reporting: invoices={inv_summ_count:,}, monthly={mc_count:,}, customer_hist={ch_count:,}, customers={distinct_customers:,}")


In [0]:
# ============================================================
# RetailRecon | Phase 7: Idempotency Test (Optimized FIFO)
# ============================================================
# Re-runs the OPTIMIZED FIFO (pre-filtered groups) and
# financial reporting to dedicated temp paths.
# Compares with validated outputs via deterministic fingerprints.
# Original outputs are NOT modified.
# ============================================================

from collections import deque
from decimal import Decimal
from hashlib import sha256
import time
import hashlib

import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DecimalType, LongType, IntegerType,
)

BASE_PATH = "/Volumes/portfolio_projects/retail_sales/raw"
LEDGER_PATH = f"{BASE_PATH}/classified_sales_ledger"

# Temp paths (cleaned up after successful comparison)
TMP_STAGING = f"{BASE_PATH}/_p7_idem_staging"
TMP_INV_SUMM = f"{BASE_PATH}/_p7_idem_inv_summ"
TMP_MC = f"{BASE_PATH}/_p7_idem_mc"

# Original outputs (read-only)
ORIG_ALLOC = f"{BASE_PATH}/cancellation_allocations_parquet"
ORIG_STATUS = f"{BASE_PATH}/cancellation_status_parquet"
ORIG_INV_SUMM = f"{BASE_PATH}/invoice_summary_parquet"
ORIG_MC = f"{BASE_PATH}/monthly_country_sales_parquet"

ZERO_DEC = F.lit(0).cast("decimal(30,4)")

results = []

def record(test_id, description, passed, actual=None, expected=None):
    status_str = "PASS" if passed else "FAIL"
    results.append((test_id, description, status_str, actual, expected))
    if passed:
        print(f"  [{status_str}] {test_id}: {description}")
    else:
        print(f"  [{status_str}] {test_id}: {description} | actual={actual} expected={expected}")

# ------------------------------------------------------------
# Self-contained FIFO function (identical to cell 10)
# ------------------------------------------------------------

_fifo_schema = StructType([
    StructField("record_type", StringType(), True),
    StructField("allocation_id", StringType(), True),
    StructField("cancellation_row_id", StringType(), True),
    StructField("sale_row_id", StringType(), True),
    StructField("cancellation_invoice", StringType(), True),
    StructField("sale_invoice", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("sale_stock_code", StringType(), True),
    StructField("cancellation_stock_code", StringType(), True),
    StructField("unit_price", DecimalType(18, 4), True),
    StructField("sale_date", TimestampType(), True),
    StructField("cancellation_date", TimestampType(), True),
    StructField("requested_quantity", LongType(), True),
    StructField("matched_quantity", LongType(), True),
    StructField("unmatched_quantity", LongType(), True),
    StructField("allocated_quantity", LongType(), True),
    StructField("allocation_value", DecimalType(30, 4), True),
    StructField("days_to_cancellation", IntegerType(), True),
    StructField("cancellation_status", StringType(), True),
    StructField("unmatched_reason", StringType(), True),
    StructField("eligible_prior_sales_quantity", LongType(), True),
])

_fifo_cols = [f.name for f in _fifo_schema.fields]
_fifo_int_cols = [
    "requested_quantity", "matched_quantity", "unmatched_quantity",
    "allocated_quantity", "days_to_cancellation",
    "eligible_prior_sales_quantity",
]
_fifo_ts_cols = ["sale_date", "cancellation_date"]


def _fifo_group(pdf):
    if pdf.empty:
        return pd.DataFrame(columns=_fifo_cols)
    pdf = pdf.copy()
    pdf["_event_priority"] = pdf["transaction_type"].map(
        {"CANCELLATION": 0, "SALE": 1}
    )
    pdf = pdf.sort_values(
        by=["invoice_date", "_event_priority", "invoice", "row_id"],
        ascending=True, kind="mergesort",
    )
    inventory = deque()
    records = []
    prior_sales_quantity = 0
    for event in pdf.to_dict("records"):
        event_type = event["transaction_type"]
        if event_type == "SALE":
            quantity = int(event["quantity"])
            if quantity <= 0:
                raise ValueError(f"Invalid sale quantity for row {event['row_id']}")
            inventory.append({
                "row_id": event["row_id"], "invoice": event["invoice"],
                "invoice_date": event["invoice_date"],
                "stock_code": event["stock_code"],
                "price": event["price"],
                "remaining_quantity": quantity,
            })
            prior_sales_quantity += quantity
            continue
        requested = abs(int(event["quantity"]))
        if requested <= 0:
            raise ValueError(f"Invalid cancellation quantity for row {event['row_id']}")
        remaining_request = requested
        matched = 0
        while remaining_request > 0 and inventory:
            while inventory and inventory[0]["remaining_quantity"] == 0:
                inventory.popleft()
            if not inventory:
                break
            lot = inventory[0]
            allocated = min(remaining_request, lot["remaining_quantity"])
            if allocated <= 0:
                raise ValueError("Non-positive FIFO allocation detected.")
            price = Decimal(str(event["price"]))
            allocation_value = (price * Decimal(allocated)).quantize(Decimal("0.0001"))
            sale_date = pd.Timestamp(lot["invoice_date"])
            cancellation_date = pd.Timestamp(event["invoice_date"])
            if sale_date >= cancellation_date:
                raise ValueError("FIFO attempted to match a sale at or after cancellation")
            pair_key = f"{lot['row_id']}||{event['row_id']}"
            records.append({
                "record_type": "ALLOCATION",
                "allocation_id": sha256(pair_key.encode("utf-8")).hexdigest(),
                "cancellation_row_id": event["row_id"],
                "sale_row_id": lot["row_id"],
                "cancellation_invoice": event["invoice"],
                "sale_invoice": lot["invoice"],
                "customer_id": event["customer_id"],
                "sale_stock_code": lot["stock_code"],
                "cancellation_stock_code": event["stock_code"],
                "unit_price": price,
                "sale_date": sale_date.to_pydatetime(),
                "cancellation_date": cancellation_date.to_pydatetime(),
                "requested_quantity": None, "matched_quantity": None,
                "unmatched_quantity": None, "allocated_quantity": allocated,
                "allocation_value": allocation_value,
                "days_to_cancellation": (cancellation_date - sale_date).days,
                "cancellation_status": None, "unmatched_reason": None,
                "eligible_prior_sales_quantity": None,
            })
            lot["remaining_quantity"] -= allocated
            remaining_request -= allocated
            matched += allocated
        unmatched = requested - matched
        if unmatched == 0:
            status, reason = "MATCHED", None
        elif matched > 0:
            status, reason = "PARTIALLY_MATCHED", "INSUFFICIENT_PRIOR_QUANTITY"
        elif prior_sales_quantity == 0:
            status, reason = "UNMATCHED", "NO_PRIOR_ELIGIBLE_SALE"
        else:
            status, reason = "UNMATCHED", "NO_AVAILABLE_PRIOR_QUANTITY"
        records.append({
            "record_type": "STATUS", "allocation_id": None,
            "cancellation_row_id": event["row_id"], "sale_row_id": None,
            "cancellation_invoice": event["invoice"], "sale_invoice": None,
            "customer_id": event["customer_id"],
            "sale_stock_code": None,
            "cancellation_stock_code": event["stock_code"],
            "unit_price": Decimal(str(event["price"])),
            "sale_date": None,
            "cancellation_date": pd.Timestamp(event["invoice_date"]).to_pydatetime(),
            "requested_quantity": requested, "matched_quantity": matched,
            "unmatched_quantity": unmatched, "allocated_quantity": None,
            "allocation_value": None, "days_to_cancellation": None,
            "cancellation_status": status, "unmatched_reason": reason,
            "eligible_prior_sales_quantity": prior_sales_quantity,
        })
    result = pd.DataFrame(records, columns=_fifo_cols)
    for column in _fifo_int_cols:
        result[column] = pd.array(result[column], dtype="Int64")
    for column in _fifo_ts_cols:
        result[column] = pd.to_datetime(result[column], errors="coerce")
    return result


def _make_null_status(cancellations_df):
    return (
        cancellations_df.filter(F.col("customer_id").isNull())
        .select(
            F.lit("STATUS").alias("record_type"),
            F.lit(None).cast("string").alias("allocation_id"),
            F.col("row_id").alias("cancellation_row_id"),
            F.lit(None).cast("string").alias("sale_row_id"),
            F.col("invoice").alias("cancellation_invoice"),
            F.lit(None).cast("string").alias("sale_invoice"),
            F.col("customer_id").cast("string").alias("customer_id"),
            F.lit(None).cast("string").alias("sale_stock_code"),
            F.col("stock_code").alias("cancellation_stock_code"),
            F.col("price").cast("decimal(18,4)").alias("unit_price"),
            F.lit(None).cast("timestamp").alias("sale_date"),
            F.col("invoice_date").cast("timestamp").alias("cancellation_date"),
            F.abs(F.col("quantity").cast("long")).alias("requested_quantity"),
            F.lit(0).cast("long").alias("matched_quantity"),
            F.abs(F.col("quantity").cast("long")).alias("unmatched_quantity"),
            F.lit(None).cast("long").alias("allocated_quantity"),
            F.lit(None).cast("decimal(30,4)").alias("allocation_value"),
            F.lit(None).cast("int").alias("days_to_cancellation"),
            F.lit("UNMATCHED").alias("cancellation_status"),
            F.lit("MISSING_CUSTOMER_ID").alias("unmatched_reason"),
            F.lit(0).cast("long").alias("eligible_prior_sales_quantity"),
        )
    )


def _alloc_fingerprint(df):
    sorted_pd = (
        df.select("allocation_id", "allocated_quantity", "cancellation_row_id", "sale_row_id")
        .orderBy("allocation_id")
        .toPandas()
    )
    return hashlib.md5(sorted_pd.to_csv(index=False).encode()).hexdigest()


def _status_fingerprint(df):
    sorted_pd = (
        df.select("cancellation_row_id", "cancellation_status", "matched_quantity", "unmatched_quantity")
        .orderBy("cancellation_row_id")
        .toPandas()
    )
    return hashlib.md5(sorted_pd.to_csv(index=False).encode()).hexdigest()


def _inv_summ_fingerprint(df):
    sorted_pd = (
        df.select("invoice", "gross_sales_value", "cancellation_value", "net_sales_value")
        .orderBy("invoice")
        .toPandas()
    )
    return hashlib.md5(sorted_pd.to_csv(index=False).encode()).hexdigest()


print("=" * 70)
print("PHASE 7: IDEMPOTENCY TEST (OPTIMIZED FIFO)")
print("=" * 70)

ledger = spark.read.parquet(LEDGER_PATH)
cancellations = ledger.filter(F.col("transaction_type") == "CANCELLATION")

# ============================================================
# D1: Rerun optimized FIFO allocation
# ============================================================
print("\n--- D1: Optimized FIFO rerun ---")
t0 = time.time()

matchable = (
    ledger.filter(F.col("customer_id").isNotNull())
    .withColumn("_match_stock_code", F.upper(F.trim(F.col("stock_code"))))
    .select(
        "row_id", "invoice", "invoice_date", "customer_id",
        "stock_code", "_match_stock_code", "price", "quantity",
        "transaction_type",
    )
)

# Pre-filter: only groups containing at least one cancellation
cancel_group_keys = (
    matchable.filter(F.col("transaction_type") == "CANCELLATION")
    .select("customer_id", "_match_stock_code", "price")
    .distinct()
)

optimized_input = matchable.join(
    cancel_group_keys,
    ["customer_id", "_match_stock_code", "price"],
    "inner",
)

grouped_results = (
    optimized_input
    .groupBy("customer_id", "_match_stock_code", "price")
    .applyInPandas(_fifo_group, schema=_fifo_schema)
)

all_results_raw = grouped_results.unionByName(_make_null_status(cancellations))

(all_results_raw.write.format("parquet").mode("overwrite")
    .option("compression", "snappy").save(TMP_STAGING))

all_results = spark.read.parquet(TMP_STAGING)

test_allocations = all_results.filter(F.col("record_type") == "ALLOCATION").select(
    "allocation_id", "cancellation_row_id", "sale_row_id",
    "cancellation_invoice", "sale_invoice", "customer_id",
    "sale_stock_code", "cancellation_stock_code", "unit_price",
    "sale_date", "cancellation_date", "allocated_quantity",
    "allocation_value", "days_to_cancellation",
)
test_status = all_results.filter(F.col("record_type") == "STATUS").select(
    "cancellation_row_id", "cancellation_invoice", "cancellation_date",
    "customer_id", "cancellation_stock_code", "unit_price",
    "requested_quantity", "matched_quantity", "unmatched_quantity",
    "cancellation_status", "unmatched_reason", "eligible_prior_sales_quantity",
)

test_alloc_count = test_allocations.count()
test_status_count = test_status.count()
fifo_elapsed = time.time() - t0
print(f"Optimized FIFO rerun: {test_alloc_count:,} allocations, {test_status_count:,} statuses ({fifo_elapsed:.1f}s)")

# ============================================================
# D2: Compare with original outputs
# ============================================================
print("\n--- D2: Fingerprint comparison ---")

orig_alloc = spark.read.parquet(ORIG_ALLOC)
orig_status = spark.read.parquet(ORIG_STATUS)

# Allocation fingerprint
orig_alloc_fp = _alloc_fingerprint(orig_alloc)
test_alloc_fp = _alloc_fingerprint(test_allocations)
record("D2a", "Allocation fingerprint match",
        orig_alloc_fp == test_alloc_fp,
        test_alloc_fp, orig_alloc_fp)

# Allocation count
orig_alloc_count = orig_alloc.count()
record("D2b", "Allocation count match",
        orig_alloc_count == test_alloc_count,
        f"{test_alloc_count}", f"{orig_alloc_count}")

# Allocation total quantity
orig_alloc_qty = orig_alloc.agg(F.coalesce(F.sum("allocated_quantity"), F.lit(0).cast("long")).alias("t")).first()["t"]
test_alloc_qty = test_allocations.agg(F.coalesce(F.sum("allocated_quantity"), F.lit(0).cast("long")).alias("t")).first()["t"]
record("D2c", "Allocation total quantity match",
        orig_alloc_qty == test_alloc_qty,
        f"{test_alloc_qty}", f"{orig_alloc_qty}")

# Allocation total value
orig_alloc_val = orig_alloc.agg(F.coalesce(F.sum("allocation_value"), ZERO_DEC).alias("t")).first()["t"]
test_alloc_val = test_allocations.agg(F.coalesce(F.sum("allocation_value"), ZERO_DEC).alias("t")).first()["t"]
record("D2d", "Allocation total value match",
        orig_alloc_val == test_alloc_val,
        str(test_alloc_val), str(orig_alloc_val))

# Status fingerprint
orig_status_fp = _status_fingerprint(orig_status)
test_status_fp = _status_fingerprint(test_status)
record("D2e", "Status fingerprint match",
        orig_status_fp == test_status_fp,
        test_status_fp, orig_status_fp)

# Status count
orig_status_count = orig_status.count()
record("D2f", "Status count match",
        orig_status_count == test_status_count,
        f"{test_status_count}", f"{orig_status_count}")

# Status totals
orig_matched_qty = orig_status.agg(F.coalesce(F.sum("matched_quantity"), F.lit(0).cast("long")).alias("t")).first()["t"]
test_matched_qty = test_status.agg(F.coalesce(F.sum("matched_quantity"), F.lit(0).cast("long")).alias("t")).first()["t"]
orig_unmatched_qty = orig_status.agg(F.coalesce(F.sum("unmatched_quantity"), F.lit(0).cast("long")).alias("t")).first()["t"]
test_unmatched_qty = test_status.agg(F.coalesce(F.sum("unmatched_quantity"), F.lit(0).cast("long")).alias("t")).first()["t"]
record("D2g", "Matched quantity total match",
        orig_matched_qty == test_matched_qty, f"{test_matched_qty}", f"{orig_matched_qty}")
record("D2h", "Unmatched quantity total match",
        orig_unmatched_qty == test_unmatched_qty, f"{test_unmatched_qty}", f"{orig_unmatched_qty}")

# No duplicate allocation IDs in rerun
test_alloc_distinct = test_allocations.select("allocation_id").distinct().count()
record("D2i", "No duplicate allocation IDs in rerun",
        test_alloc_distinct == test_alloc_count,
        f"distinct={test_alloc_distinct}, total={test_alloc_count}", "equal")

# No inflated totals: rerun totals do not exceed original
total_not_inflated = test_alloc_count <= orig_alloc_count and test_status_count <= orig_status_count
record("D2j", "No inflated counts",
        total_not_inflated,
        f"alloc={test_alloc_count}<= {orig_alloc_count}, status={test_status_count}<= {orig_status_count}",
        "not exceeded")

# ============================================================
# D3: Rerun financial reporting
# ============================================================
print("\n--- D3: Financial reporting rerun ---")
t1 = time.time()

test_inv_summ = (
    ledger.groupBy("invoice").agg(
        F.min("invoice_date").alias("invoice_date"),
        F.max("country").alias("country"),
        F.sum(F.when(F.col("transaction_type") == "SALE", F.col("signed_line_value")).otherwise(ZERO_DEC)).alias("gross_sales_value"),
        F.sum(F.when(F.col("transaction_type") == "CANCELLATION", F.abs(F.col("signed_line_value"))).otherwise(ZERO_DEC)).alias("cancellation_value"),
        F.sum(F.when(F.col("transaction_type") == "SALE", F.lit(1)).otherwise(F.lit(0))).alias("sales_line_count"),
        F.sum(F.when(F.col("transaction_type") == "CANCELLATION", F.lit(1)).otherwise(F.lit(0))).alias("cancellation_line_count"),
        F.sum(F.when(F.col("transaction_type") == "SALE", F.col("quantity").cast("long")).otherwise(F.lit(0))).alias("gross_sold_quantity"),
        F.sum(F.when(F.col("transaction_type") == "CANCELLATION", F.abs(F.col("quantity").cast("long"))).otherwise(F.lit(0))).alias("abs_cancelled_quantity"),
    )
    .withColumn("net_sales_value", (F.col("gross_sales_value") - F.col("cancellation_value")).cast("decimal(30,4)"))
)

test_inv_summ_count = test_inv_summ.count()
report_elapsed = time.time() - t1
print(f"Invoice summary rerun: {test_inv_summ_count:,} records ({report_elapsed:.1f}s)")

# ============================================================
# D4: Compare financial reporting
# ============================================================
print("\n--- D4: Financial reporting comparison ---")

orig_inv_summ = spark.read.parquet(ORIG_INV_SUMM)
orig_mc = spark.read.parquet(ORIG_MC)

# Invoice summary fingerprint
orig_inv_fp = _inv_summ_fingerprint(orig_inv_summ)
test_inv_fp = _inv_summ_fingerprint(test_inv_summ)
record("D4a", "Invoice summary fingerprint match",
        orig_inv_fp == test_inv_fp, test_inv_fp, orig_inv_fp)

# Invoice summary count
orig_inv_count = orig_inv_summ.count()
record("D4b", "Invoice summary count match",
        orig_inv_count == test_inv_summ_count,
        f"{test_inv_summ_count}", f"{orig_inv_count}")

# Financial totals match
orig_gross = orig_inv_summ.agg(F.sum("gross_sales_value")).first()[0]
test_gross = test_inv_summ.agg(F.sum("gross_sales_value")).first()[0]
orig_cancel_val = orig_inv_summ.agg(F.sum("cancellation_value")).first()[0]
test_cancel_val = test_inv_summ.agg(F.sum("cancellation_value")).first()[0]
record("D4c", "Gross total match",
        orig_gross == test_gross, str(test_gross), str(orig_gross))
record("D4d", "Cancellation total match",
        orig_cancel_val == test_cancel_val, str(test_cancel_val), str(orig_cancel_val))

# No inflated totals
total_gross_not_inflated = float(str(test_gross)) <= float(str(orig_gross)) * 1.0001
total_cancel_not_inflated = float(str(test_cancel_val)) <= float(str(orig_cancel_val)) * 1.0001
record("D4e", "No inflated financial totals",
        total_gross_not_inflated and total_cancel_not_inflated,
        f"gross={test_gross}, cancel={test_cancel_val}", "not inflated")

# No duplicate invoice keys in rerun
test_inv_dupes = test_inv_summ.groupBy("invoice").count().filter(F.col("count") > 1).count()
record("D4f", "No duplicate invoice keys in rerun",
        test_inv_dupes == 0, f"dupes={test_inv_dupes}", "dupes=0")

# ============================================================
# D5: Cleanup temp artifacts
# ============================================================
print("\n--- D5: Cleanup ---")
for path in [TMP_STAGING, TMP_INV_SUMM, TMP_MC]:
    try:
        dbutils.fs.rm(path, recurse=True)
        print(f"  Cleaned: {path}")
    except Exception as e:
        print(f"  Skip: {path} ({e})")

# ============================================================
# Summary
# ============================================================
print("\n" + "=" * 70)
print("PHASE 7 IDEMPOTENCY TEST SUMMARY")
print("=" * 70)
passed = sum(1 for r in results if r[2] == "PASS")
failed = sum(1 for r in results if r[2] == "FAIL")
print(f"Total tests: {len(results)}")
print(f"Passed: {passed}")
print(f"Failed: {failed}")

if failed > 0:
    print("\nFAILED TESTS:")
    for r in results:
        if r[2] == "FAIL":
            print(f"  {r[0]}: {r[1]} | actual={r[3]} expected={r[4]}")

print(f"\n{'ALL IDEMPOTENCY TESTS PASSED' if failed == 0 else 'IDEMPOTENCY TEST FAILED'}")
print(f"Timing: optimized FIFO={fifo_elapsed:.1f}s, reporting={report_elapsed:.1f}s")
print("Original outputs were not modified.")
